# NYC Listings Scout — Discovery Only

This notebook is the **discovery stage only**.

It should:
- collect candidate listings from the active sources
- apply the current constraints:
  - max monthly rent: **$4,500**
  - area: **Hell's Kitchen / Chelsea / Hudson Yards / Lincoln Center**
  - fallback box: **8th–12th Ave, 20th–62nd St**
  - duration: **roughly 3–6 months**
- classify rows into:
  - `exact_address_hit`
  - `contact_queue`
  - `manual_review_or_contact`
  - `parser_review`
  - `skip_not_goal`

It should **not**:
- send inquiries
- try to unlock addresses by messaging
- perform the Craigslist address follow-up pass
- mix email/contact logic into the scrape


In [ ]:
# ═══════════════════════════════════════
# Cell 1: Install dependencies
# ═══════════════════════════════════════
!pip install -q requests beautifulsoup4 lxml pandas
!pip install -q playwright
!playwright install chromium --with-deps
print('✅ All dependencies installed')


In [ ]:
# ═══════════════════════════════════════
# Cell 2: Configuration + Utilities
# ═══════════════════════════════════════
import os, csv, json, re, time, random, requests
from datetime import datetime
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

ALL_RESULTS = []  # Global accumulator

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

CSV_FIELDS = [
    "source", "title", "price_raw", "price_num", "price_period",
    "est_monthly", "neighborhood", "borough", "bedrooms",
    "address", "apt_num", "furnished", "listing_type", "poster_type",
    "dates", "amenities", "building_clues", "description", "url", "scraped_at",
    "looks_sublet", "looks_short_term", "goal_match",
    "address_found", "needs_contact_for_address", "needs_manual_review", "address_confidence",
    "search_price_match", "search_area_match", "search_duration_match", "search_pass",
    "duration_months_min", "duration_months_max", "search_fail_reasons",
    "site_active", "site_address_mode", "action_bucket",
]

# Goal controls
ONLY_GOAL_MATCHES = True
REQUIRE_EXACT_ADDRESS_FOR_HIT = True
SEND_INQUIRIES = False   # discovery notebook never sends inquiries
MAX_INQUIRIES_PER_RUN = 0

# Search constraints requested by user
SEARCH_MAX_MONTHLY = 4500
SEARCH_MIN_MONTHS = 3
SEARCH_MAX_MONTHS = 6
TARGET_AREA_LABEL = "Hell's Kitchen / Chelsea / Hudson Yards / Lincoln Center"
TARGET_STREET_MIN = 20
TARGET_STREET_MAX = 62
TARGET_AVENUE_MIN = 8
TARGET_AVENUE_MAX = 12
TARGET_NEIGHBORHOOD_PATTERNS = [
    r"hell'?s\s+kitchen", r"\bclinton\b(?!\s+st)", r"\bchelsea\b", r"hudson\s+yards",
    r"lincoln\s+center", r"columbus\s+circle", r"midtown\s+west",
    r"theater\s+district", r"theatre\s+district", r"west\s+chelsea",
    r"\bmidtown\b", r"\buws\b", r"upper\s+west\s+side",
    r"central\s+park\s+west", r"\bcpw\b", r"\briverside\b",
]

SOURCE_POLICIES = {
    "Craigslist":       {"active": True,  "address_mode": "mixed",         "target_bias": "always"},
    "LeaseBreak":       {"active": True,  "address_mode": "direct",        "target_bias": "always"},
    "SpareRoom":        {"active": True,  "address_mode": "mixed",         "target_bias": "always"},
    "Sublet.com":       {"active": True,  "address_mode": "mixed",         "target_bias": "always"},
    "SabbaticalHomes":  {"active": True,  "address_mode": "contact_first", "target_bias": "always"},
    "Zumper":           {"active": True,  "address_mode": "direct",        "target_bias": "keyword"},
    "Loftey":           {"active": True,  "address_mode": "direct",        "target_bias": "keyword"},
    "StreetEasy":       {"active": False, "address_mode": "direct",        "target_bias": "off"},
    "Ohana":            {"active": True,  "address_mode": "direct",        "target_bias": "always"},
    "June Homes":       {"active": True,  "address_mode": "direct",        "target_bias": "always"},
    "RentHop":          {"active": True,  "address_mode": "direct",        "target_bias": "keyword"},
    "Listings Project": {"active": True,  "address_mode": "contact_first", "target_bias": "always"},
}

SHORT_TERM_PATTERNS = [
    r"short\s*term", r"\bsublet\b", r"\bsublease\b", r"\btemporary\b", r"month[-\s]*to[-\s]*month",
    r"\bmonthly\b", r"\bweekly\b", r"\bdaily\b", r"flex[-\s]*lease", r"\bfurnished\b", r"available\s+through",
    r"available\s+until", r"lease\s+takeover", r"lease\s+assignment", r"summer\s+sublet",
]
HARD_EXCLUDE_PATTERNS = [
    r"\b12\s*month", r"\b12-month", r"annual lease", r"long[-\s]*term only", r"year lease",
    r"no short term", r"minimum\s+12\s+months",
]
ADDRESS_RE = re.compile(
    r"\b(\d{1,5}\s+(?:[A-Za-z0-9.'-]+\s+){0,6}"
    r"(?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Place|Pl|Court|Ct|Lane|Ln|Drive|Dr|Broadway|Parkway|Pkwy)"
    r"(?:\s+(?:North|South|East|West|N|S|E|W))?"
    r"(?:\s*(?:#|Apt\.?|Apartment|Unit)\s*[A-Za-z0-9-]+)?)\b",
    flags=re.I,
)
MONTH_NAME_TO_NUM = {
    'january': 1, 'jan': 1,
    'february': 2, 'feb': 2,
    'march': 3, 'mar': 3,
    'april': 4, 'apr': 4,
    'may': 5,
    'june': 6, 'jun': 6,
    'july': 7, 'jul': 7,
    'august': 8, 'aug': 8,
    'september': 9, 'sep': 9, 'sept': 9,
    'october': 10, 'oct': 10,
    'november': 11, 'nov': 11,
    'december': 12, 'dec': 12,
}
WORD_NUMBERS = {
    'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6,
    'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10, 'eleven': 11, 'twelve': 12,
}

pw = None
browser = None
ctx = None
page = None

def polite_sleep(lo=2, hi=5):
    time.sleep(random.uniform(lo, hi))

def now_iso():
    return datetime.utcnow().isoformat() + 'Z'

def normalize_space(text):
    text = str(text or "")
    text = text.replace('\xa0', '').replace('\u200b', '').replace('\u200c', '').replace('\u200d', '').replace('\ufeff', '')
    return re.sub(r"\s+", " ", text).strip()

def parse_price(text):
    if not text:
        return None, "", None
    text = str(text).replace('\xa0', '').replace('\u200b', '').replace('\u200c', '').replace('\u200d', '').replace('\ufeff', '')
    m = re.search(r'\$\s*([\d,]+)\s*(?:/\s*(\w+))?', text)
    if not m:
        return None, "", None
    num = int(m.group(1).replace(",", ""))
    period = (m.group(2) or "month").lower()
    if period.startswith("mo"):
        period = "month"
    elif period.startswith("w"):
        period = "week"
    elif period.startswith("d") or period.startswith("ni"):
        period = "day"
    est = num
    if period == "week":
        est = num * 4
    elif period == "day":
        est = num * 30
    return num, period, est

def detect_beds(text):
    t = (text or "").lower()
    if "studio" in t:
        return "Studio"
    if re.search(r"\b4[\s-]?(?:bed|br|bedroom)", t):
        return "4BR+"
    if re.search(r"\b3[\s-]?(?:bed|br|bedroom)", t):
        return "3BR"
    if re.search(r"\b2[\s-]?(?:bed|br|bedroom)", t):
        return "2BR"
    if re.search(r"\b1[\s-]?(?:bed|br|bedroom)|one bed", t):
        return "1BR"
    if re.search(r"\broom\b", t):
        return "Room"
    return ""

def detect_furnished(t):
    t = (t or "").lower()
    if "unfurnished" in t:
        return "No"
    if "furnished" in t:
        return "Yes"
    return ""

def detect_amenities(t):
    t = (t or "").lower()
    a = []
    if re.search(r"washer|w/d|laundry", t):
        a.append("Laundry")
    if "dishwasher" in t:
        a.append("Dishwasher")
    if re.search(r"outdoor|backyard|garden|patio|terrace|rooftop|balcon", t):
        a.append("Outdoor")
    if re.search(r"doorman|concierge", t):
        a.append("Doorman")
    if re.search(r"\belevator\b", t):
        a.append("Elevator")
    if re.search(r"\bgym\b|fitness", t):
        a.append("Gym")
    if re.search(r"renovated", t):
        a.append("Renovated")
    if re.search(r"pre-?war", t):
        a.append("Pre-war")
    if re.search(r"hardwood", t):
        a.append("Hardwood")
    if re.search(r"exposed brick", t):
        a.append("Exposed brick")
    return "; ".join(a)

def detect_building(t):
    t = (t or "").lower()
    c = []
    if re.search(r"doorman|concierge", t):
        c.append("Doorman")
    if re.search(r"\belevator\b", t):
        c.append("Elevator")
    if re.search(r"walk[\s-]?up", t):
        c.append("Walk-up")
    if re.search(r"pre[\s-]?war", t):
        c.append("Pre-war")
    if re.search(r"brownstone", t):
        c.append("Brownstone")
    if re.search(r"rent[\s-]?stabili", t):
        c.append("Rent-stabilized")
    m = re.search(r"(\d+)(?:st|nd|rd|th)\s+floor\b", t)
    if m:
        c.append(f"Floor {m.group(1)}")
    if "top floor" in t:
        c.append("Top floor")
    return "; ".join(c)

def universal_extract(html, base_url=''):
    """Fallback extractor: finds all <a> tags near price patterns."""
    soup = BeautifulSoup(html, 'lxml')
    results = []
    seen_urls = set()
    for a in soup.find_all('a', href=True):
        href = a['href']
        if not href.startswith('http') and base_url:
            href = base_url.rstrip('/') + '/' + href.lstrip('/')
        parent = a.find_parent(['div', 'li', 'article', 'section', 'tr'])
        if not parent:
            continue
        pt = parent.get_text(' ', strip=True)
        if len(pt) < 20 or len(pt) > 2000:
            continue
        pm = re.search(r'\$[\d,]+', pt)
        if not pm:
            continue
        if href in seen_urls:
            continue
        seen_urls.add(href)
        link_text = a.get_text(strip=True)
        results.append({
            'url': href,
            'title': link_text[:200],
            'card_text': pt[:500],
            'price_found': pm.group(),
        })
    return results

def extract_exact_address(text):
    text = normalize_space(text)
    m = ADDRESS_RE.search(text)
    if not m:
        return ""
    addr = m.group(1).strip(" ,.;")
    addr = re.sub(r"\bSt\b", "St", addr)
    addr = re.sub(r"\bAve\b", "Ave", addr)
    return addr

def source_policy(source_name):
    return SOURCE_POLICIES.get(
        source_name or "",
        {"active": True, "address_mode": "mixed", "target_bias": "keyword"}
    )

def word_or_int_to_num(token):
    if token is None:
        return None
    token = str(token).strip().lower()
    if token.isdigit():
        return int(token)
    return WORD_NUMBERS.get(token)

def extract_duration_window_months(text):
    text = normalize_space(text).lower()
    if not text:
        return None, None, 'unknown'

    num_token = r'(\d{1,2}|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)'

    m = re.search(rf'(?:minimum|min)?\s*(?:stay\s+of\s+)?{num_token}\s*(?:-|to|through|–|—)\s*{num_token}\s*months?', text)
    if m:
        a = word_or_int_to_num(m.group(1))
        b = word_or_int_to_num(m.group(2))
        if a is not None and b is not None:
            return min(a, b), max(a, b), 'explicit_range'

    m = re.search(rf'(?:for|about|around|roughly|approx(?:imately)?)\s+{num_token}\s*(?:-|to|through|–|—)\s*{num_token}\s*months?', text)
    if m:
        a = word_or_int_to_num(m.group(1))
        b = word_or_int_to_num(m.group(2))
        if a is not None and b is not None:
            return min(a, b), max(a, b), 'explicit_range'

    m = re.search(rf'(?:minimum|min)\s*{num_token}\+?\s*months?', text)
    if m:
        a = word_or_int_to_num(m.group(1))
        if a is not None:
            return a, None, 'minimum_only'

    m = re.search(rf'(?:maximum|max|up to)\s*{num_token}\s*months?', text)
    if m:
        b = word_or_int_to_num(m.group(1))
        if b is not None:
            return None, b, 'maximum_only'

    m = re.search(rf'\b{num_token}\+?\s*months?\b', text)
    if m:
        a = word_or_int_to_num(m.group(1))
        if a is not None:
            if '+' in m.group(0):
                return a, None, 'exact_plus'
            return a, a, 'exact_months'

    month_hits = []
    for mm in re.finditer(r'\b(jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:t|tember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)\b', text):
        month_hits.append(MONTH_NAME_TO_NUM[mm.group(1)])
    if len(month_hits) >= 2:
        start = month_hits[0]
        end = month_hits[1]
        diff = end - start
        if diff <= 0:
            diff += 12
        if 1 <= diff <= 12:
            return diff, diff, 'month_name_range'

    if re.search(r'month[-\s]*to[-\s]*month|flex[-\s]*lease|flexible term', text):
        return None, None, 'flex_unspecified'

    return None, None, 'unknown'

def duration_matches_target(text):
    dmin, dmax, reason = extract_duration_window_months(text)
    target_min, target_max = SEARCH_MIN_MONTHS, SEARCH_MAX_MONTHS
    if dmin is None and dmax is None:
        # No duration info or flex — treat as potentially compatible
        return True, dmin, dmax, reason
    low = dmin if dmin is not None else target_min
    high = dmax if dmax is not None else target_max
    overlap = not (high < target_min or low > target_max)
    return overlap, dmin, dmax, reason

def text_mentions_target_neighborhood(text):
    text = normalize_space(text).lower()
    if not text:
        return False
    return any(re.search(p, text) for p in TARGET_NEIGHBORHOOD_PATTERNS)

def extract_street_numbers(text):
    text = normalize_space(text).lower()
    nums = []
    for m in re.finditer(r'\b(?:w(?:est)?\s*)?(\d{1,3})(?:st|nd|rd|th)?\s+(?:street|st)\b', text):
        try:
            nums.append(int(m.group(1)))
        except Exception:
            pass
    return nums

def extract_avenue_numbers(text):
    text = normalize_space(text).lower()
    nums = []
    for m in re.finditer(r'\b(8|9|10|11|12)(?:th)?\s+(?:avenue|ave)\b', text):
        try:
            nums.append(int(m.group(1)))
        except Exception:
            pass
    for word, num in [('eighth', 8), ('ninth', 9), ('tenth', 10), ('eleventh', 11), ('twelfth', 12)]:
        if re.search(rf'\b{word}\s+(?:avenue|ave)\b', text):
            nums.append(num)
    return sorted(set(nums))

def location_matches_target(record):
    text = normalize_space(' '.join([
        record.get('title', ''), record.get('neighborhood', ''), record.get('address', ''),
        record.get('description', ''), record.get('dates', ''), record.get('borough', '')
    ]))
    if text_mentions_target_neighborhood(text):
        return True, 'neighborhood_keyword'

    streets = extract_street_numbers(text)
    avenues = extract_avenue_numbers(text)
    street_ok = any(TARGET_STREET_MIN <= s <= TARGET_STREET_MAX for s in streets)
    avenue_ok = any(TARGET_AVENUE_MIN <= a <= TARGET_AVENUE_MAX for a in avenues)

    if street_ok and avenue_ok:
        return True, 'street_and_avenue_box'
    if street_ok and re.search(r'\bwest\b|\bw\s*\d{1,3}(?:st|nd|rd|th)?\s+st', text.lower()):
        return True, 'west_street_range'

    hood = normalize_space(record.get('neighborhood', '')).lower()
    if hood and text_mentions_target_neighborhood(hood):
        return True, 'neighborhood_field'

    return False, ''

def price_matches_target(record):
    est = record.get('est_monthly')
    if est is None:
        pn, pp, em = parse_price(record.get('price_raw', ''))
        if em is not None:
            record['price_num'] = record.get('price_num') or pn
            record['price_period'] = record.get('price_period') or pp
            record['est_monthly'] = em
            est = em
    if est is None:
        return False, 'unknown_price'
    if est < 400:
        return False, f'price_too_low={est}'
    return est <= SEARCH_MAX_MONTHLY, f'est_monthly={est}'

def looks_like_goal_listing(record):
    policy = source_policy(record.get('source'))
    source = record.get('source') or ""

    text = normalize_space(" ".join([
        record.get('title', ''), record.get('listing_type', ''), record.get('description', ''),
        record.get('dates', ''), source, record.get('url', '')
    ])).lower()

    if not policy.get('active', True):
        return False, False

    if any(re.search(p, text) for p in HARD_EXCLUDE_PATTERNS):
        return False, False

    keyword_short = any(re.search(p, text) for p in SHORT_TERM_PATTERNS)
    bias = policy.get('target_bias', 'keyword')

    if bias == 'always':
        looks_short = True
        looks_sublet = True
    elif bias == 'keyword':
        looks_short = keyword_short
        looks_sublet = keyword_short
    else:
        looks_short = False
        looks_sublet = False

    return looks_sublet, looks_short

def enrich_listing(record):
    r = dict(record)
    policy = source_policy(r.get('source'))
    text = normalize_space(" ".join([
        r.get('title', ''), r.get('description', ''), r.get('listing_type', ''),
        r.get('dates', ''), r.get('address', ''), r.get('neighborhood', ''), r.get('borough', '')
    ]))
    if not r.get('furnished'):
        r['furnished'] = detect_furnished(text)
    if not r.get('amenities'):
        r['amenities'] = detect_amenities(text)
    if not r.get('building_clues'):
        r['building_clues'] = detect_building(text)
    if not r.get('bedrooms'):
        r['bedrooms'] = detect_beds(text)

    inferred = extract_exact_address(text)
    if inferred and not r.get('address'):
        r['address'] = inferred

    looks_sublet, looks_short = looks_like_goal_listing(r)
    r['looks_sublet'] = 'Yes' if looks_sublet else 'No'
    r['looks_short_term'] = 'Yes' if looks_short else 'No'

    address_found = bool(normalize_space(r.get('address', '')))
    address_mode = policy.get('address_mode', 'mixed')
    active = bool(policy.get('active', True))

    price_ok, price_reason = price_matches_target(r)
    area_ok, area_reason = location_matches_target(r)
    duration_ok, dmin, dmax, duration_reason = duration_matches_target(text)

    base_goal = active and looks_sublet and looks_short
    search_pass = base_goal and price_ok and area_ok and duration_ok

    fail_reasons = []
    if not active:
        fail_reasons.append('source_disabled')
    if not looks_sublet or not looks_short:
        fail_reasons.append('not_short_term_signal')
    if not price_ok:
        fail_reasons.append(price_reason)
    if not area_ok:
        fail_reasons.append('outside_target_area')
    if not duration_ok:
        fail_reasons.append(duration_reason or 'duration_mismatch')

    if not search_pass:
        action_bucket = 'skip_not_goal'
    elif address_found:
        action_bucket = 'exact_address_hit'
    elif address_mode == 'contact_first':
        action_bucket = 'contact_queue'
    elif address_mode == 'mixed':
        action_bucket = 'manual_review_or_contact'
    else:
        action_bucket = 'parser_review'

    r['goal_match'] = 'Yes' if search_pass else 'No'
    r['search_pass'] = 'Yes' if search_pass else 'No'
    r['search_price_match'] = 'Yes' if price_ok else 'No'
    r['search_area_match'] = 'Yes' if area_ok else 'No'
    r['search_duration_match'] = 'Yes' if duration_ok else 'No'
    r['duration_months_min'] = dmin
    r['duration_months_max'] = dmax
    r['search_fail_reasons'] = '; '.join([x for x in fail_reasons if x])
    r['address_found'] = 'Yes' if address_found else 'No'
    r['address_confidence'] = 'high' if address_found else ''
    r['site_active'] = 'Yes' if active else 'No'
    r['site_address_mode'] = address_mode
    r['action_bucket'] = action_bucket
    r['needs_contact_for_address'] = 'Yes' if action_bucket in {'contact_queue', 'manual_review_or_contact'} else 'No'
    r['needs_manual_review'] = 'Yes' if action_bucket in {'manual_review_or_contact', 'parser_review'} else 'No'
    return r

def write_results(rows, filename='listings.csv'):
    path = OUTPUT_DIR / filename
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS, extrasaction='ignore')
        w.writeheader()
        w.writerows(rows)
    print(f'  -> {len(rows)} rows -> {path}')
    return path

def deduplicate(rows):
    from collections import Counter
    seen = set()
    out = []
    url_counts = Counter((r.get('url') or '').strip().rstrip('/') for r in rows)
    for r in rows:
        url = (r.get('url') or '').strip().rstrip('/')
        if not url or url_counts.get(url, 0) > 1:
            key = (
                normalize_space(r.get('source', '')),
                normalize_space(r.get('title', '')),
                normalize_space(r.get('price_raw', '')),
                normalize_space(r.get('address', '')),
                normalize_space(r.get('dates', '')),
            )
        else:
            key = url
        if key in seen:
            continue
        seen.add(key)
        out.append(r)
    return out

def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def ensure_playwright_browser(headless=True):
    global pw, browser, ctx, page
    try:
        if browser is not None and browser.is_connected() and ctx is not None:
            return browser, ctx, page
    except Exception:
        pass

    from playwright.sync_api import sync_playwright

    pw = sync_playwright().start()
    browser = pw.chromium.launch(
        headless=headless,
        args=[
            '--no-sandbox',
            '--disable-setuid-sandbox',
            '--disable-dev-shm-usage',
            '--disable-gpu',
            '--no-first-run',
            '--no-zygote',
        ]
    )
    ctx = browser.new_context(
        viewport={'width': 1440, 'height': 900},
        user_agent=HEADERS['User-Agent'],
    )
    ctx.set_default_timeout(25000)
    page = ctx.new_page()
    return browser, ctx, page

def safe_close_page(pg):
    try:
        if pg is not None:
            pg.close()
    except Exception:
        pass

def close_playwright_browser():
    global pw, browser, ctx, page
    safe_close_page(page)
    page = None
    ctx = None
    try:
        if browser is not None:
            browser.close()
    except Exception:
        pass
    browser = None
    try:
        if pw is not None:
            pw.stop()
    except Exception:
        pass
    pw = None

# DOM / route preflight controls
PREFLIGHT_ENABLED = True
PREFLIGHT_SKIP_FAILING_SOURCES = True
PREFLIGHT_SKIP_DEGRADED_SOURCES = False
PREFLIGHT_REQUEST_TIMEOUT = 20
PREFLIGHT_WAIT_MS = 4000
PREFLIGHT_RESULTS = []
SOURCE_HEALTH = {
    name: {'status': 'UNTESTED', 'stage': '', 'details': '', 'artifacts': []}
    for name in SOURCE_POLICIES
}
PREFLIGHT_ARTIFACT_DIR = OUTPUT_DIR / 'preflight_artifacts'
PREFLIGHT_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def slugify(value):
    value = normalize_space(str(value or '')).lower()
    value = re.sub(r'[^a-z0-9]+', '_', value).strip('_')
    return value or 'item'

def preflight_save_text(source, stage, text, ext='html'):
    path = PREFLIGHT_ARTIFACT_DIR / f"{slugify(source)}__{slugify(stage)}.{ext}"
    with open(path, 'w', encoding='utf-8') as f:
        f.write(text or '')
    return str(path)

def preflight_save_bytes(source, stage, data, ext='bin'):
    path = PREFLIGHT_ARTIFACT_DIR / f"{slugify(source)}__{slugify(stage)}.{ext}"
    with open(path, 'wb') as f:
        f.write(data or b'')
    return str(path)

def record_preflight(source, status, stage, details='', url='', artifacts=None):
    artifacts = artifacts or []
    rank = {'UNTESTED': 0, 'PASS': 1, 'DEGRADED': 2, 'FAIL': 3}
    current = SOURCE_HEALTH.get(source, {'status': 'UNTESTED', 'stage': '', 'details': '', 'artifacts': []})
    if rank.get(status, 0) >= rank.get(current.get('status', 'UNTESTED'), 0):
        SOURCE_HEALTH[source] = {
            'status': status,
            'stage': stage,
            'details': details,
            'artifacts': artifacts,
        }
    PREFLIGHT_RESULTS.append({
        'source': source,
        'status': status,
        'stage': stage,
        'details': details,
        'url': url,
        'artifacts': '; '.join(artifacts),
        'checked_at': now_iso(),
    })
    return SOURCE_HEALTH[source]

def site_preflight_status(source):
    return (SOURCE_HEALTH.get(source) or {}).get('status', 'UNTESTED')

def site_preflight_ok(source):
    if not PREFLIGHT_ENABLED:
        return True
    status = site_preflight_status(source)
    if status == 'FAIL' and PREFLIGHT_SKIP_FAILING_SOURCES:
        return False
    if status == 'DEGRADED' and PREFLIGHT_SKIP_DEGRADED_SOURCES:
        return False
    return True

def export_preflight_results():
    summary_rows = []
    for source in sorted(SOURCE_POLICIES):
        info = SOURCE_HEALTH.get(source, {})
        summary_rows.append({
            'source': source,
            'status': info.get('status', 'UNTESTED'),
            'stage': info.get('stage', ''),
            'details': info.get('details', ''),
            'artifacts': '; '.join(info.get('artifacts', []) or []),
        })
    summary_path = OUTPUT_DIR / 'nyc_preflight_summary.csv'
    detail_path = OUTPUT_DIR / 'nyc_preflight_details.csv'
    json_path = OUTPUT_DIR / 'nyc_preflight_summary.json'
    pd.DataFrame(summary_rows).to_csv(summary_path, index=False)
    pd.DataFrame(PREFLIGHT_RESULTS).to_csv(detail_path, index=False)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(summary_rows, f, indent=2)
    return summary_path, detail_path, json_path

print(f'✅ Config loaded. Output -> {OUTPUT_DIR.absolute()}')
print('Goal mode -> short-term / sublet only with strict filters')
print(f'  Max monthly: ${SEARCH_MAX_MONTHLY:,}')
print(f'  Target area: {TARGET_AREA_LABEL}')
print(f'  Streets: {TARGET_STREET_MIN}-{TARGET_STREET_MAX}')
print(f'  Avenues: {TARGET_AVENUE_MIN}-{TARGET_AVENUE_MAX}')
print(f'  Duration target: {SEARCH_MIN_MONTHS}-{SEARCH_MAX_MONTHS} months')
print('Active source policies:')
for _src, _policy in SOURCE_POLICIES.items():
    state = 'ACTIVE' if _policy['active'] else 'DISABLED'
    print(f"  {_src:>16}: {state} | {_policy['address_mode']} | bias={_policy['target_bias']}")


In [ ]:
# ═══════════════════════════════════════
# Cell 3: Inline Credentials + Mount Drive
# ═══════════════════════════════════════
# Credentials are stored directly in this notebook instead of Colab Secrets.
# Keep this notebook private and do not upload it to any public repo.

CREDS = {
    'lp':     {'email': 'maniacidante@gmail.com',         'password': 'bokquk-cAgzax-7nomty'},
    'lb':     {'email': 'caballerodante421@gmail.com',    'password': 'ratqa2-gacnAg-qurwyd'},
    'sr':     {'email': 'caballerodante421@gmail.com',    'password': 'pewpoh-dyckyb-3qAwfe'},
    'sc':     {'email': 'caballerodante421@gmail.com',    'password': 'zajhop-5ricne-Timman'},
    'sh':     {'email': 'Caballerodante421@gmail.com',    'password': 'sufba1-wozreb-Hedcug'},
    'zumper': {'email': 'caballerodante421@gmail.com',    'password': 'misCuj-zezbuj-6saggo'},
    'loftey': {'email': 'caballerodante421@gmail.com',    'password': 'ziqwe7-mytgor-sIbbeh'},
    'ohana':  {'email': 'caballerodante421@gmail.com',    'password': '7r2_D$pbvLJsDSB'},
    'jh':     {'email': 'caballerodante421@gmail.com',    'password': 'xojcyg-7sorje-cezziF'},
}

def mask_email(email):
    if not email or '@' not in email:
        return 'NOT SET'
    local, domain = email.split('@', 1)
    if len(local) <= 2:
        masked_local = local[0] + '*'
    else:
        masked_local = local[:2] + '*' * max(1, len(local) - 2)
    return f'{masked_local}@{domain}'

# Verify all loaded without printing full credentials
for key, cred in CREDS.items():
    status = '✅' if cred['email'] and cred['password'] else '❌ MISSING'
    print(f'  {key:>8}: {status} — {mask_email(cred["email"])}')

# Drive mount / export directory
SAVE_TO_DRIVE = True
DRIVE_DIR = OUTPUT_DIR / 'drive_exports'

if SAVE_TO_DRIVE:
    if running_in_colab():
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            DRIVE_DIR = Path('/content/drive/MyDrive/NYC_Listings')
        except Exception as e:
            print(f'ℹ️ Drive mount skipped: {e}')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f'\n✅ Export dir → {DRIVE_DIR}')

In [ ]:
# ═══════════════════════════════════════
# Cell 4: Authenticated discovery helpers
# ═══════════════════════════════════════
AUTHENTICATED_DISCOVERY = True
AUTH_ARTIFACT_DIR = OUTPUT_DIR / 'auth_artifacts'
AUTH_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
AUTH_SESSION_LOG = {}

def auth_save_artifacts(source, pg, stage='auth_failure'):
    artifacts = []
    try:
        html = pg.content()
        html_path = AUTH_ARTIFACT_DIR / f"{slugify(source)}__{slugify(stage)}.html"
        html_path.write_text(html or '', encoding='utf-8')
        artifacts.append(str(html_path))
    except Exception:
        pass
    try:
        png_path = AUTH_ARTIFACT_DIR / f"{slugify(source)}__{slugify(stage)}.png"
        pg.screenshot(path=str(png_path), full_page=True)
        artifacts.append(str(png_path))
    except Exception:
        pass
    return artifacts

def confirm_logged_in(pg, source, success_selectors=None, success_text=None, login_markers=None):
    success_selectors = success_selectors or []
    success_text = success_text or []
    login_markers = login_markers or ['/login', 'login', 'signin', 'sign-in', 'logon']

    try:
        url = (pg.url or '').lower()
    except Exception:
        url = ''
    try:
        body = pg.locator('body').inner_text(timeout=3000) or ''
    except Exception:
        body = ''

    try:
        has_password = bool(pg.query_selector('input[type="password"]'))
    except Exception:
        has_password = False
    try:
        has_login_input = bool(pg.query_selector('input[type="email"], input[name*="email" i], input[name*="user" i], input[name*="login" i]'))
    except Exception:
        has_login_input = False

    selector_hit = False
    for sel in success_selectors:
        try:
            if pg.query_selector(sel):
                selector_hit = True
                break
        except Exception:
            continue

    body_l = body.lower()
    text_hit = any(t.lower() in body_l for t in success_text)
    still_on_login = any(m in url for m in login_markers)
    negative_only = not still_on_login and not has_password and not has_login_input
    if negative_only and not selector_hit and not text_hit:
        print(f'    \u26a0\ufe0f {source}: no positive auth signal, relying on negative test')
    looks_logged_in = selector_hit or text_hit or negative_only

    AUTH_SESSION_LOG[source] = {
        'ok': bool(looks_logged_in),
        'url': url,
        'selector_hit': selector_hit,
        'text_hit': text_hit,
        'has_password': has_password,
        'has_login_input': has_login_input,
        'checked_at': now_iso(),
    }
    return bool(looks_logged_in)

def require_logged_in(pg, source, success_selectors=None, success_text=None, stage='post_login'):
    if not AUTHENTICATED_DISCOVERY:
        return True
    ok = confirm_logged_in(pg, source, success_selectors=success_selectors, success_text=success_text)
    if ok:
        print(f'    ✅ Auth confirmed for {source}')
        return True
    artifacts = auth_save_artifacts(source, pg, stage=stage)
    raise RuntimeError(f'{source} login not confirmed; artifacts=' + '; '.join(artifacts))

print('✅ Authenticated discovery helpers loaded')



## Launch Playwright

In [ ]:
# ═══════════════════════════════════════
# Cell 8: Launch Playwright (Colab-safe)
# ═══════════════════════════════════════
ensure_playwright_browser(headless=True)
print('✅ Playwright browser ready (headless, no-sandbox)')

## Preflight

In [ ]:
# ═══════════════════════════════════════
# Preflight DOM / route health check
# ═══════════════════════════════════════
# Purpose:
# - verify key login fields / route structure before the main scrape
# - capture HTML + screenshots for broken sources
# - mark sources PASS / DEGRADED / FAIL
# - allow later cells to auto-skip explicit FAILs

from IPython.display import display

if not PREFLIGHT_ENABLED:
    print('ℹ️ Preflight disabled in config.')
else:
    ensure_playwright_browser(headless=True)

    PREFLIGHT_RESULTS.clear()
    for _src in SOURCE_POLICIES:
        SOURCE_HEALTH[_src] = {'status': 'UNTESTED', 'stage': '', 'details': '', 'artifacts': []}

    def _contains_antibot(text):
        text = (text or '').lower()
        anti = [
            'verify you are human', 'access denied', 'captcha', 'robot or human',
            'are you human', 'unusual traffic', 'security check', 'cloudflare',
            'forbidden', 'temporarily blocked'
        ]
        return any(x in text for x in anti)

    def _preflight_http(source, stage, url, must_find_any=None, must_find_all=None, min_body_chars=250):
        must_find_any = must_find_any or []
        must_find_all = must_find_all or []
        artifacts = []
        try:
            r = requests.get(url, headers=HEADERS, timeout=PREFLIGHT_REQUEST_TIMEOUT)
            text = r.text or ''
            artifacts.append(preflight_save_text(source, f'{stage}__http', text, 'html'))
            status = 'PASS'
            notes = [f'http={r.status_code}', f'chars={len(text)}']
            if r.status_code >= 400:
                status = 'FAIL'
                notes.append('http_error')
            elif len(text) < min_body_chars:
                status = 'DEGRADED'
                notes.append('body_too_short')
            if _contains_antibot(text):
                status = 'FAIL'
                notes.append('anti_bot_detected')
            if must_find_all:
                missing_all = [p for p in must_find_all if not re.search(p, text, re.I | re.S)]
                if missing_all:
                    status = 'DEGRADED' if status == 'PASS' else status
                    notes.append('missing_all=' + ' | '.join(missing_all[:3]))
            if must_find_any and not any(re.search(p, text, re.I | re.S) for p in must_find_any):
                status = 'DEGRADED' if status == 'PASS' else status
                notes.append('missing_any_group')
            record_preflight(source, status, stage, '; '.join(notes), url, artifacts)
            print(f'  {source:>18} | {stage:<18} | {status:<8} | {"; ".join(notes[:3])}')
        except Exception as e:
            record_preflight(source, 'FAIL', stage, f'exception={e}', url, artifacts)
            print(f'  {source:>18} | {stage:<18} | FAIL     | exception={e}')

    def _preflight_browser(source, stage, url, selector_groups=None, must_find_any=None,
                           must_find_all=None, min_body_chars=250, min_anchor_count=2,
                           wait_until='domcontentloaded', wait_ms=None):
        selector_groups = selector_groups or []
        must_find_any = must_find_any or []
        must_find_all = must_find_all or []
        wait_ms = PREFLIGHT_WAIT_MS if wait_ms is None else wait_ms
        artifacts = []
        pg = None
        try:
            pg = ctx.new_page()
            resp = pg.goto(url, timeout=35000, wait_until=wait_until)
            pg.wait_for_timeout(wait_ms)
            html = pg.content()
            body = ''
            try:
                body = pg.locator('body').inner_text(timeout=3000)
            except Exception:
                body = ''
            artifacts.append(preflight_save_text(source, f'{stage}__browser', html, 'html'))
            try:
                png = pg.screenshot(full_page=True)
                artifacts.append(preflight_save_bytes(source, f'{stage}__browser', png, 'png'))
            except Exception:
                pass

            status_code = getattr(resp, 'status', None)
            anchor_count = 0
            try:
                anchor_count = pg.eval_on_selector_all('a[href]', 'els => els.length')
            except Exception:
                anchor_count = 0

            notes = [f'http={status_code}', f'chars={len(body)}', f'anchors={anchor_count}']
            status = 'PASS'

            if status_code and status_code >= 400:
                status = 'FAIL'
                notes.append('http_error')
            if _contains_antibot(body) or _contains_antibot(html):
                status = 'FAIL'
                notes.append('anti_bot_detected')

            missing_groups = []
            for label, selectors in selector_groups:
                matched = False
                for sel in selectors:
                    try:
                        if pg.query_selector(sel):
                            matched = True
                            break
                    except Exception:
                        continue
                if not matched:
                    missing_groups.append(label)

            if missing_groups:
                status = 'DEGRADED' if status == 'PASS' else status
                notes.append('missing_selectors=' + ','.join(missing_groups))

            if len(body) < min_body_chars:
                status = 'DEGRADED' if status == 'PASS' else status
                notes.append('body_too_short')

            if min_anchor_count and anchor_count < min_anchor_count:
                status = 'DEGRADED' if status == 'PASS' else status
                notes.append('low_anchor_count')

            missing_all = [p for p in must_find_all if not re.search(p, body or html, re.I | re.S)]
            if missing_all:
                status = 'DEGRADED' if status == 'PASS' else status
                notes.append('missing_all=' + ' | '.join(missing_all[:3]))

            if must_find_any and not any(re.search(p, body or html, re.I | re.S) for p in must_find_any):
                status = 'DEGRADED' if status == 'PASS' else status
                notes.append('missing_any_group')

            record_preflight(source, status, stage, '; '.join(notes), url, artifacts)
            print(f'  {source:>18} | {stage:<18} | {status:<8} | {"; ".join(notes[:4])}')
        except Exception as e:
            record_preflight(source, 'FAIL', stage, f'exception={e}', url, artifacts)
            print(f'  {source:>18} | {stage:<18} | FAIL     | exception={e}')
        finally:
            safe_close_page(pg)

    print('Running preflight checks...')
    _preflight_http(
        'Craigslist', 'rss_feed',
        'https://newyork.craigslist.org/search/mnh/sub?format=rss&max_price=4500&hasPic=1',
        must_find_any=[r'<item\b', r'<rss\b', r'<rdf:RDF\b', r'craigslist']
    )

    _preflight_browser(
        'LeaseBreak', 'login', 'https://www.leasebreak.com/login',
        selector_groups=[
            ('email', ['input[type="email"]', 'input[name="email"]', 'input[name="username"]']),
            ('password', ['input[type="password"]']),
        ],
        must_find_any=[r'log\s*in', r'sign\s*in'],
        min_body_chars=120, min_anchor_count=1
    )
    _preflight_browser(
        'LeaseBreak', 'inventory', 'https://www.leasebreak.com/short-term-rentals-nyc',
        must_find_any=[r'\$', r'short\s*term', r'sublet', r'furnished', r'manhattan', r'new york'],
        min_body_chars=400, min_anchor_count=10
    )

    _preflight_browser(
        'SpareRoom', 'login', 'https://www.spareroom.com/logon/',
        selector_groups=[
            ('email', ['input[name="loginemail"]', 'input[type="email"]', 'input[name="email"]']),
            ('password', ['input[name="loginpassword"]', 'input[type="password"]']),
        ],
        must_find_any=[r'log\s*in', r'spare\s*room'],
        min_body_chars=120, min_anchor_count=1
    )
    _preflight_browser(
        'SpareRoom', 'inventory', 'https://www.spareroom.com/rooms-for-rent/new-york?page=1',
        must_find_any=[r'\$', r'room', r'new york', r'month'],
        min_body_chars=350, min_anchor_count=10
    )

    _preflight_browser(
        'Sublet.com', 'login', 'https://www.sublet.com/login',
        selector_groups=[
            ('email', ['input[type="email"]', 'input[name="email"]', 'input[name="username"]', 'input[name="login"]']),
            ('password', ['input[type="password"]']),
        ],
        must_find_any=[r'log\s*in', r'sign\s*in', r'sublet'],
        min_body_chars=120, min_anchor_count=1
    )
    _preflight_browser(
        'Sublet.com', 'inventory', 'https://www.sublet.com/new-york-city',
        must_find_any=[r'\$', r'sublet', r'new york', r'month'],
        min_body_chars=300, min_anchor_count=10
    )

    _preflight_browser(
        'SabbaticalHomes', 'login', 'https://www.sabbaticalhomes.com/Login',
        selector_groups=[
            ('email', ['input[type="email"]', 'input[name="email"]', 'input[name="username"]', 'input[name="Email"]']),
            ('password', ['input[type="password"]']),
        ],
        must_find_any=[r'log\s*in', r'sabbatical'],
        min_body_chars=120, min_anchor_count=1
    )
    _preflight_browser(
        'SabbaticalHomes', 'inventory',
        'https://www.sabbaticalhomes.com/Home-Exchange-Rental-House-Sitting-Search?field_1=New+York&field_2=NY&field_3=USA&miles=5&lat=40.7725&lng=-73.9835',
        must_find_any=[r'new york', r'rental', r'contact', r'home'],
        min_body_chars=300, min_anchor_count=10
    )

    _preflight_browser(
        'Zumper', 'login', 'https://www.zumper.com/login',
        selector_groups=[
            ('email', ['input[type="email"]', 'input[name="email"]', 'input[placeholder*="email" i]']),
            ('password', ['input[type="password"]']),
        ],
        must_find_any=[r'log\s*in', r'zumper'],
        min_body_chars=120, min_anchor_count=1
    )
    _preflight_browser(
        'Zumper', 'inventory', 'https://www.zumper.com/apartments-for-rent/new-york-ny/short-term',
        must_find_any=[r'\$', r'short\s*term', r'new york', r'apartment'],
        min_body_chars=350, min_anchor_count=10,
        wait_until='networkidle', wait_ms=5000
    )

    _preflight_browser(
        'Loftey', 'login', 'https://loftey.com/login',
        selector_groups=[
            ('email', ['input[type="email"]', 'input[name="email"]', 'input[name="username"]', 'input[placeholder*="email" i]']),
            ('password', ['input[type="password"]']),
        ],
        must_find_any=[r'log\s*in', r'sign\s*in', r'loftey'],
        min_body_chars=120, min_anchor_count=1
    )
    _preflight_browser(
        'Loftey', 'inventory', 'https://loftey.com/apartments-for-rent/manhattan/chelsea',
        must_find_any=[r'\$', r'chelsea', r'rent', r'apartment'],
        min_body_chars=300, min_anchor_count=10
    )

    _preflight_browser(
        'Ohana', 'login', 'https://liveohana.ai/login',
        selector_groups=[
            ('email', ['input[type="email"]', 'input[name="email"]', 'input[placeholder*="email" i]']),
            ('password', ['input[type="password"]']),
        ],
        must_find_any=[r'log\s*in', r'sign\s*in', r'ohana'],
        min_body_chars=120, min_anchor_count=1,
        wait_ms=5000
    )
    _preflight_browser(
        'Ohana', 'inventory', 'https://liveohana.ai/sublet/new-york-city',
        must_find_any=[r'\$', r'sublet', r'new york', r'room', r'apartment'],
        min_body_chars=250, min_anchor_count=5,
        wait_until='networkidle', wait_ms=6000
    )

    _preflight_browser(
        'June Homes', 'login', 'https://junehomes.com/login',
        selector_groups=[
            ('email', ['input[type="email"]', 'input[name="email"]', 'input[placeholder*="email" i]']),
            ('password', ['input[type="password"]']),
        ],
        must_find_any=[r'log\s*in', r'sign\s*in', r'june'],
        min_body_chars=120, min_anchor_count=1
    )
    _preflight_browser(
        'June Homes', 'inventory', 'https://junehomes.com/residences/new-york-city-ny',
        must_find_any=[r'\$', r'new york', r'residence', r'furnished', r'room'],
        min_body_chars=300, min_anchor_count=8,
        wait_until='networkidle', wait_ms=6000
    )

    _preflight_http(
        'RentHop', 'inventory', 'https://www.renthop.com/apartments-for-rent/new-york-ny/sublet',
        must_find_any=[r'/listings/', r'sublet', r'\$', r'renthop'],
        min_body_chars=300
    )

    _preflight_browser(
        'Listings Project', 'login', 'https://www.listingsproject.com/login',
        selector_groups=[
            ('email', ['input[type="email"]', 'input[name="email"]']),
            ('password', ['input[type="password"]']),
        ],
        must_find_any=[r'log\s*in', r'listings\s*project'],
        min_body_chars=120, min_anchor_count=1
    )
    _preflight_browser(
        'Listings Project', 'inventory', 'https://www.listingsproject.com/real-estate/new-york-city/sublets',
        must_find_any=[r'new york city', r'sublet', r'read the full listing', r'contact'],
        min_body_chars=250, min_anchor_count=8
    )

    summary_path, detail_path, json_path = export_preflight_results()
    summary_df = pd.DataFrame([
        {
            'source': s,
            'status': info.get('status', 'UNTESTED'),
            'stage': info.get('stage', ''),
            'details': info.get('details', ''),
        }
        for s, info in sorted(SOURCE_HEALTH.items())
    ])
    print(f'\nSaved preflight artifacts to: {PREFLIGHT_ARTIFACT_DIR}')
    print(f'Summary CSV: {summary_path}')
    print(f'Detail CSV:  {detail_path}')
    display(summary_df)

## 🔵 Craigslist (RSS — no auth needed)


In [ ]:
if PREFLIGHT_ENABLED and not site_preflight_ok('Craigslist'):
    print("⏭ Craigslist skipped due to preflight status: " + site_preflight_status('Craigslist'))
    cl_rows = []
else:
    # ═══════════════════════════════════════
    # Cell 4: Craigslist via RSS (most reliable)
    # ═══════════════════════════════════════
    from xml.etree import ElementTree

    CL_FEEDS = [
        ("Manhattan", "https://newyork.craigslist.org/search/mnh/sub?format=rss&max_price=4500&hasPic=1"),
    ]

    cl_rows = []
    for boro, feed_url in CL_FEEDS:
        print(f'  Fetching {boro} RSS...')
        try:
            r = requests.get(feed_url, headers=HEADERS, timeout=15)
            r.raise_for_status()
            print(f'    Status {r.status_code}, {len(r.content)} bytes')

            # CL RSS uses RDF namespace
            root = ElementTree.fromstring(r.content)
            # Try multiple namespace patterns (CL has changed formats over the years)
            items = root.findall('.//{http://purl.org/rss/1.0/}item')
            if not items:
                items = root.findall('.//item')
            if not items:
                items = root.findall('.//channel/item')
            print(f'    Found {len(items)} RSS items')

            for item in items:
                # Helper to get text from either namespaced or plain tag
                def gt(tag):
                    for prefix in ['{http://purl.org/rss/1.0/}', '']:
                        el = item.find(prefix + tag)
                        if el is not None and el.text:
                            return el.text.strip()
                    return ''

                title = gt('title')
                link = gt('link')
                desc_html = gt('description')
                if not link:
                    continue

                # Parse description HTML
                desc_soup = BeautifulSoup(desc_html, 'lxml') if desc_html else None
                desc_text = desc_soup.get_text(' ', strip=True) if desc_soup else ''

                # CL title format: "$2,500 / 1br - Sunny Apt (Upper West Side)"
                pn, pp, em = parse_price(title)
                hood_m = re.search(r'\(([^)]+)\)\s*$', title)
                hood = hood_m.group(1) if hood_m else ''
                clean_title = re.sub(r'^\$[\d,]+\s*/\s*\w+\s*-\s*', '', title).strip()
                clean_title = re.sub(r'\s*\([^)]+\)\s*$', '', clean_title).strip()

                full = title + ' ' + desc_text
                cl_rows.append({
                    'source': 'Craigslist',
                    'title': clean_title or title[:200],
                    'price_raw': f'${pn:,}/{pp}' if pn else '',
                    'price_num': pn, 'price_period': pp, 'est_monthly': em,
                    'neighborhood': hood, 'borough': boro,
                    'bedrooms': detect_beds(full),
                    'furnished': detect_furnished(full),
                    'listing_type': 'Sublet',
                    'poster_type': 'Likely Tenant',
                    'amenities': detect_amenities(full),
                    'building_clues': detect_building(full),
                    'description': desc_text[:300],
                    'url': link,
                    'scraped_at': now_iso(),
                })

        except ElementTree.ParseError:
            # RSS might not be available; fall back to HTML
            print(f'    RSS parse failed. Trying HTML fallback...')
            try:
                params = feed_url.split('?')[1] if '?' in feed_url else ''
                params = re.sub(r'&?format=rss&?', '&', params).strip('&')
                html_url = feed_url.split('?')[0] + ('?' + params if params else '')
                r2 = requests.get(html_url, headers=HEADERS, timeout=15)
                raw = universal_extract(r2.text, 'https://newyork.craigslist.org')
                print(f'    HTML fallback found {len(raw)} link+price combos')
                for item in raw[:60]:
                    pn, pp, em = parse_price(item['price_found'])
                    cl_rows.append({
                        'source': 'Craigslist', 'title': item['title'][:200],
                        'price_raw': item['price_found'],
                        'price_num': pn, 'price_period': pp or 'month', 'est_monthly': em,
                        'borough': boro, 'listing_type': 'Sublet',
                        'poster_type': 'Likely Tenant',
                        'url': item['url'], 'scraped_at': now_iso(),
                        'description': item['card_text'][:300],
                    })
            except Exception as e2:
                print(f'    HTML fallback also failed: {e2}')

        except Exception as e:
            print(f'    Error: {e}')

        polite_sleep(2, 4)

    ALL_RESULTS.extend(cl_rows)
    print(f'\n✅ Craigslist: {len(cl_rows)} listings')
    if cl_rows:
        print(f'   Sample: {cl_rows[0]["title"][:60]} — {cl_rows[0]["price_raw"]}')


## 🟢 LeaseBreak (authenticated)


In [ ]:
if PREFLIGHT_ENABLED and not site_preflight_ok('LeaseBreak'):
    print("⏭ LeaseBreak skipped due to preflight status: " + site_preflight_status('LeaseBreak'))
    lb_rows = []
else:
    # ═══════════════════════════════════════
    # Cell 5: LeaseBreak (authenticated)
    # ═══════════════════════════════════════
    LB_BASE = 'https://www.leasebreak.com'

    ensure_playwright_browser()

    print('  Logging into LeaseBreak...')
    lb_rows = []
    try:
        pg_lb = ctx.new_page()
        pg_lb.goto(f'{LB_BASE}/login', timeout=30000, wait_until='domcontentloaded')
        pg_lb.wait_for_timeout(3000)

        # Fill login form
        email_in = pg_lb.query_selector('input[type="email"], input[name="email"], input[name="username"]')
        pass_in = pg_lb.query_selector('input[type="password"]')
        if not email_in:
            inputs = pg_lb.query_selector_all('input')
            for inp in inputs:
                t = inp.get_attribute('type') or ''
                n = inp.get_attribute('name') or ''
                if t in ('text','email') or 'email' in n or 'user' in n:
                    email_in = inp; break
        if email_in and pass_in:
            email_in.fill(CREDS['lb']['email'])
            pg_lb.wait_for_timeout(300)
            pass_in.fill(CREDS['lb']['password'])
            pg_lb.wait_for_timeout(300)
            submit = pg_lb.query_selector('button[type="submit"], input[type="submit"]') or pg_lb.query_selector('button')
            if submit: submit.click()
            else: pass_in.press('Enter')
            pg_lb.wait_for_timeout(4000)
            print(f'    Logged in. URL: {pg_lb.url[:60]}')
            require_logged_in(pg_lb, 'LeaseBreak', success_selectors=['a[href*="logout"]', 'a[href*="account"]', 'a:has-text("Logout")', 'a:has-text("My Account")'], success_text=['logout', 'my account', 'saved searches'], stage='leasebreak_login')
        else:
            raise RuntimeError('LeaseBreak login form not found')

        # Scrape verified short-term/sublet entry points (not generic apartment pages)
        LB_SEARCHES = [
            ('NYC Sublets', f'{LB_BASE}/sublets-nyc'),
            ('NYC Short Term', f'{LB_BASE}/short-term-rentals-nyc'),
        ]
        for area_name, url in LB_SEARCHES:
            print(f'  LeaseBreak {area_name}...')
            pg_lb.goto(url, timeout=20000, wait_until='domcontentloaded')
            pg_lb.wait_for_timeout(3000)

            items = pg_lb.evaluate("""
            () => {
                const results = [];
                const seen = new Set();
                document.querySelectorAll('a[href]').forEach(a => {
                    const href = a.href;
                    if (seen.has(href)) return;
                    let card = a.closest('div, li, article, tr');
                    if (!card) return;
                    const text = card.innerText || '';
                    if (text.length < 30 || text.length > 2000) return;
                    const pm = text.match(/\\$[\\d,]+/);
                    if (!pm) return;
                    // Filter to listing-like URLs
                    if (!href.includes('leasebreak.com') || href.includes('/login') || href.includes('/about')) return;
                    seen.add(href);
                    results.push({url: href, title: a.innerText.trim().substring(0,200), card_text: text.substring(0,500), price: pm[0]});
                });
                return results;
            }
            """)
            # Filter to actual listing links (not nav)
            items = [x for x in items if len(x['card_text']) > 80]
            print(f'    Found {len(items)} listings')

            for item in items:
                pn, pp, em = parse_price(item['price'])
                card = item['card_text']
                lb_rows.append({
                    'source': 'LeaseBreak', 'title': item['title'][:200],
                    'price_raw': item['price'],
                    'price_num': pn, 'price_period': pp or 'month', 'est_monthly': em,
                    'neighborhood': area_name, 'bedrooms': detect_beds(card),
                    'furnished': detect_furnished(card),
                    'listing_type': 'Sublet/Lease Break',
                    'poster_type': 'Mixed — verify',
                    'amenities': detect_amenities(card),
                    'building_clues': detect_building(card),
                    'description': card[:300], 'url': item['url'], 'scraped_at': now_iso(),
                })
            polite_sleep(3, 5)

    except Exception as e:
        print(f'  ❌ LeaseBreak error: {e}')
    finally:
        safe_close_page(pg_lb)

    ALL_RESULTS.extend(lb_rows)
    print(f'\n✅ LeaseBreak: {len(lb_rows)} listings')



## 🟡 SpareRoom (authenticated)


In [ ]:
if PREFLIGHT_ENABLED and not site_preflight_ok('SpareRoom'):
    print("⏭ SpareRoom skipped due to preflight status: " + site_preflight_status('SpareRoom'))
    sr_rows = []
else:
    # ═══════════════════════════════════════
    # Cell 6: SpareRoom (authenticated, Playwright)
    # ═══════════════════════════════════════
    ensure_playwright_browser()

    print('  Logging into SpareRoom...')
    sr_rows = []
    try:
        pg_sr = ctx.new_page()
        pg_sr.goto('https://www.spareroom.com/logon/', timeout=30000, wait_until='domcontentloaded')
        pg_sr.wait_for_timeout(3000)

        email_in = pg_sr.query_selector('input[name="loginemail"], input[type="email"], input[name="email"]')
        pass_in = pg_sr.query_selector('input[name="loginpassword"], input[type="password"]')
        if not email_in or not pass_in:
            inputs = pg_sr.query_selector_all('input')
            for inp in inputs:
                t = (inp.get_attribute('type') or '').lower()
                n = (inp.get_attribute('name') or '').lower()
                if not email_in and (t == 'email' or 'email' in n): email_in = inp
                if not pass_in and t == 'password': pass_in = inp

        if email_in and pass_in:
            email_in.fill(CREDS['sr']['email'])
            pg_sr.wait_for_timeout(300)
            pass_in.fill(CREDS['sr']['password'])
            pg_sr.wait_for_timeout(300)
            submit = pg_sr.query_selector('button[type="submit"], input[type="submit"], button:has-text("Log in")')
            if submit: submit.click()
            else: pass_in.press('Enter')
            pg_sr.wait_for_timeout(4000)
            print(f'    Logged in. URL: {pg_sr.url[:60]}')
            require_logged_in(pg_sr, 'SpareRoom', success_selectors=['a[href*="logout"]', 'a:has-text("Log out")', 'a:has-text("My account")'], success_text=['log out', 'my account', 'saved ads'], stage='spareroom_login')
        else:
            raise RuntimeError('SpareRoom login form not found')

        for pg_num in range(1, 4):
            url = f'https://www.spareroom.com/rooms-for-rent/new-york?page={pg_num}'
            print(f'  SpareRoom page {pg_num}...')
            pg_sr.goto(url, timeout=20000, wait_until='domcontentloaded')
            pg_sr.wait_for_timeout(3000)

            items = pg_sr.evaluate("""
            () => {
                const results = [];
                const seen = new Set();
                document.querySelectorAll('a[href]').forEach(a => {
                    const href = a.href;
                    if (seen.has(href)) return;
                    if (!href.includes('spareroom.com')) return;
                    if (!href.includes('/rooms-for-rent/') && !href.includes('/flat-share/')) return;
                    if (href.length < 40) return;
                    let card = a.closest('li, div, article');
                    if (!card) return;
                    const text = card.innerText || '';
                    if (text.length < 40 || text.length > 2000) return;
                    const pm = text.match(/\\$[\\d,]+|£[\\d,]+/);
                    if (!pm) return;
                    seen.add(href);
                    results.push({url: href, title: a.innerText.trim().substring(0,200), card_text: text.substring(0,500), price: pm[0]});
                });
                return results;
            }
            """)
            print(f'    Found {len(items)} listings')

            for item in items:
                pn, pp, em = parse_price(item['price'])
                card = item['card_text']
                sr_rows.append({
                    'source': 'SpareRoom', 'title': item['title'][:200],
                    'price_raw': item['price'],
                    'price_num': pn, 'price_period': pp or 'month', 'est_monthly': em,
                    'bedrooms': detect_beds(card), 'furnished': detect_furnished(card),
                    'listing_type': 'Room/Sublet', 'poster_type': 'Mixed — verify',
                    'description': card[:300], 'url': item['url'], 'scraped_at': now_iso(),
                })
            polite_sleep(3, 5)

    except Exception as e:
        print(f'  ❌ SpareRoom error: {e}')
    finally:
        safe_close_page(pg_sr)

    ALL_RESULTS.extend(sr_rows)
    print(f'\n✅ SpareRoom: {len(sr_rows)} listings')



## 🟠 Sublet.com (authenticated)


In [ ]:
if PREFLIGHT_ENABLED and not site_preflight_ok('Sublet.com'):
    print("⏭ Sublet.com skipped due to preflight status: " + site_preflight_status('Sublet.com'))
    sc_rows = []
else:
    # ═══════════════════════════════════════
    # Cell 7a: Sublet.com (authenticated, Playwright)
    # ═══════════════════════════════════════
    ensure_playwright_browser()

    print('  Logging into Sublet.com...')
    sc_rows = []
    try:
        pg_sc = ctx.new_page()
        pg_sc.goto('https://www.sublet.com/login', timeout=30000, wait_until='domcontentloaded')
        pg_sc.wait_for_timeout(3000)

        email_in = pg_sc.query_selector('input[type="email"], input[name="email"], input[name="username"], input[name="login"]')
        pass_in = pg_sc.query_selector('input[type="password"]')
        if not email_in:
            for inp in pg_sc.query_selector_all('input'):
                t = (inp.get_attribute('type') or '').lower()
                n = (inp.get_attribute('name') or '').lower()
                if t in ('text','email') or 'email' in n or 'user' in n or 'login' in n:
                    email_in = inp; break

        if email_in and pass_in:
            email_in.fill(CREDS['sc']['email'])
            pg_sc.wait_for_timeout(300)
            pass_in.fill(CREDS['sc']['password'])
            pg_sc.wait_for_timeout(300)
            submit = pg_sc.query_selector('button[type="submit"], input[type="submit"], button:has-text("Log"), button:has-text("Sign")')
            if submit: submit.click()
            else: pass_in.press('Enter')
            pg_sc.wait_for_timeout(4000)
            print(f'    Logged in. URL: {pg_sc.url[:60]}')
            require_logged_in(pg_sc, 'Sublet.com', success_selectors=['a[href*="logout"]', 'a:has-text("Logout")', 'a:has-text("My Account")'], success_text=['logout', 'my account', 'favorites'], stage='sublet_login')
        else:
            raise RuntimeError('Sublet.com login form not found')

        # Scrape NYC sublets
        pg_sc.goto('https://www.sublet.com/new-york-city', timeout=20000, wait_until='domcontentloaded')
        pg_sc.wait_for_timeout(3000)

        items = pg_sc.evaluate("""
        () => {
            const results = [];
            const seen = new Set();
            document.querySelectorAll('a[href*="/property/"]').forEach(a => {
                const href = a.href;
                if (seen.has(href)) return;
                seen.add(href);
                let card = a.closest('div, li, article') || a.parentElement;
                const text = (card || a).innerText || '';
                const pm = text.match(/\\$[\\d,]+/);
                if (pm) results.push({url: href, title: a.innerText.trim().substring(0,200), card_text: text.substring(0,500), price: pm[0]});
            });
            return results;
        }
        """)
        print(f'    Found {len(items)} property links')

        for item in items:
            pn, pp, em = parse_price(item['price'])
            card = item['card_text']
            sc_rows.append({
                'source': 'Sublet.com', 'title': item['title'][:200] or 'Sublet.com listing',
                'price_raw': item['price'],
                'price_num': pn, 'price_period': pp or 'month', 'est_monthly': em,
                'bedrooms': detect_beds(card), 'furnished': detect_furnished(card),
                'listing_type': 'Sublet', 'poster_type': 'Mixed — verify',
                'description': card[:300], 'url': item['url'], 'scraped_at': now_iso(),
            })

    except Exception as e:
        print(f'  ❌ Sublet.com error: {e}')
    finally:
        safe_close_page(pg_sc)

    ALL_RESULTS.extend(sc_rows)
    print(f'\n✅ Sublet.com: {len(sc_rows)} listings')



## 🎓 SabbaticalHomes (authenticated)


In [ ]:
if PREFLIGHT_ENABLED and not site_preflight_ok('SabbaticalHomes'):
    print("⏭ SabbaticalHomes skipped due to preflight status: " + site_preflight_status('SabbaticalHomes'))
    sh_rows = []
else:
    # ═══════════════════════════════════════
    # Cell 7b: SabbaticalHomes (authenticated, Playwright)
    # ═══════════════════════════════════════
    ensure_playwright_browser()

    print('  Logging into SabbaticalHomes...')
    sh_rows = []
    try:
        pg_sh = ctx.new_page()
        pg_sh.goto('https://www.sabbaticalhomes.com/Login', timeout=30000, wait_until='domcontentloaded')
        pg_sh.wait_for_timeout(3000)

        email_in = pg_sh.query_selector('input[type="email"], input[name="email"], input[name="username"], input[name="Email"]')
        pass_in = pg_sh.query_selector('input[type="password"]')
        if not email_in:
            for inp in pg_sh.query_selector_all('input'):
                t = (inp.get_attribute('type') or '').lower()
                n = (inp.get_attribute('name') or '').lower()
                p = (inp.get_attribute('placeholder') or '').lower()
                if t in ('text','email') or 'email' in n or 'email' in p or 'user' in n:
                    email_in = inp; break

        if email_in and pass_in:
            email_in.fill(CREDS['sh']['email'])
            pg_sh.wait_for_timeout(300)
            pass_in.fill(CREDS['sh']['password'])
            pg_sh.wait_for_timeout(300)
            submit = pg_sh.query_selector('button[type="submit"], input[type="submit"], button:has-text("Log"), button:has-text("Sign")')
            if submit: submit.click()
            else: pass_in.press('Enter')
            pg_sh.wait_for_timeout(5000)
            print(f'    Logged in. URL: {pg_sh.url[:60]}')
            require_logged_in(pg_sh, 'SabbaticalHomes', success_selectors=['a[href*="logout"]', 'a:has-text("Logout")', 'a:has-text("My Account")'], success_text=['logout', 'my account', 'dashboard'], stage='sabbaticalhomes_login')
        else:
            raise RuntimeError('SabbaticalHomes login form not found')

        # Search NYC area - target zone only
        search_urls = [
            ('Lincoln Center', 'https://www.sabbaticalhomes.com/Home-Exchange-Rental-House-Sitting-Search?field_1=New+York&field_2=NY&field_3=USA&miles=5&lat=40.7725&lng=-73.9835'),
        ]

        for area_name, url in search_urls:
            print(f'  SabbaticalHomes {area_name}...')
            try:
                pg_sh.goto(url, timeout=20000, wait_until='domcontentloaded')
                pg_sh.wait_for_timeout(3000)

                items = pg_sh.evaluate("""
                () => {
                    const results = [];
                    const seen = new Set();
                    document.querySelectorAll('a[href]').forEach(a => {
                        const href = a.href;
                        if (seen.has(href)) return;
                        if (!href.includes('sabbaticalhomes.com')) return;
                        if (!href.includes('/rental/') && !href.includes('/listing/') && !href.includes('/Home-')) return;
                        if (href.includes('/Login') || href.includes('/Search')) return;
                        let card = a.closest('div, li, article, tr') || a.parentElement;
                        if (!card) return;
                        const text = card.innerText || '';
                        if (text.length < 30 || text.length > 2000) return;
                        const pm = text.match(/\\$[\\d,]+/);
                        seen.add(href);
                        results.push({url: href, title: a.innerText.trim().substring(0,200), card_text: text.substring(0,500), price: pm ? pm[0] : ''});
                    });
                    return results;
                }
                """)
                print(f'    Found {len(items)} listing links')

                for item in items:
                    pn, pp, em = parse_price(item['price'])
                    card = item['card_text']
                    sh_rows.append({
                        'source': 'SabbaticalHomes', 'title': item['title'][:200] or f'SabbaticalHomes {area_name}',
                        'price_raw': item['price'] or 'See listing',
                        'price_num': pn, 'price_period': pp or 'month', 'est_monthly': em,
                        'neighborhood': area_name, 'borough': 'Manhattan',
                        'bedrooms': detect_beds(card), 'furnished': detect_furnished(card),
                        'listing_type': 'Sublet (academic)',
                        'poster_type': 'Tenant (academic)',
                        'description': card[:300], 'url': item['url'], 'scraped_at': now_iso(),
                    })
            except Exception as e:
                print(f'    ❌ {e}')
            polite_sleep(3, 5)

    except Exception as e:
        print(f'  ❌ SabbaticalHomes error: {e}')
    finally:
        safe_close_page(pg_sh)

    ALL_RESULTS.extend(sh_rows)
    print(f'\n✅ SabbaticalHomes: {len(sh_rows)} listings')



## 🏠 Zumper (authenticated)


In [ ]:
if PREFLIGHT_ENABLED and not site_preflight_ok('Zumper'):
    print("⏭ Zumper skipped due to preflight status: " + site_preflight_status('Zumper'))
    zm_rows = []
else:
    # ═══════════════════════════════════════
    # Cell 7c: Zumper (authenticated, Playwright)
    # ═══════════════════════════════════════
    ensure_playwright_browser()

    print('  Logging into Zumper...')
    zm_rows = []
    try:
        pg_zm = ctx.new_page()
        pg_zm.goto('https://www.zumper.com/login', timeout=30000, wait_until='domcontentloaded')
        pg_zm.wait_for_timeout(3000)

        email_in = pg_zm.query_selector('input[type="email"], input[name="email"], input[placeholder*="email" i]')
        pass_in = pg_zm.query_selector('input[type="password"]')
        if not email_in:
            for inp in pg_zm.query_selector_all('input'):
                t = (inp.get_attribute('type') or '').lower()
                n = (inp.get_attribute('name') or '').lower()
                p = (inp.get_attribute('placeholder') or '').lower()
                if t in ('text','email') or 'email' in n or 'email' in p:
                    email_in = inp; break

        if email_in and pass_in:
            email_in.fill(CREDS['zumper']['email'])
            pg_zm.wait_for_timeout(300)
            pass_in.fill(CREDS['zumper']['password'])
            pg_zm.wait_for_timeout(300)
            submit = pg_zm.query_selector('button[type="submit"], input[type="submit"], button:has-text("Log"), button:has-text("Sign")')
            if submit: submit.click()
            else: pass_in.press('Enter')
            pg_zm.wait_for_timeout(5000)
            print(f'    Logged in. URL: {pg_zm.url[:60]}')
            require_logged_in(pg_zm, 'Zumper', success_selectors=['a[href*="logout"]', 'button:has-text("Log out")', 'a:has-text("Account")'], success_text=['log out', 'account settings', 'favorites'], stage='zumper_login')
        else:
            raise RuntimeError('Zumper login form not found')

        # Zumper NYC search pages
        ZM_SEARCHES = [
            ('NYC Short Term', 'https://www.zumper.com/apartments-for-rent/new-york-ny/short-term'),
            ('Manhattan Short Term', 'https://www.zumper.com/apartments-for-rent/manhattan-ny/short-term'),
        ]

        for hood, url in ZM_SEARCHES:
            print(f'  Zumper {hood}...')
            try:
                pg_zm.goto(url, timeout=20000, wait_until='domcontentloaded')
                pg_zm.wait_for_timeout(4000)

                items = pg_zm.evaluate("""
                () => {
                    const results = [];
                    const seen = new Set();
                    document.querySelectorAll('a[href]').forEach(a => {
                        const href = a.href;
                        if (seen.has(href)) return;
                        if (!href.includes('zumper.com')) return;
                        if (!href.includes('/address/') && !href.includes('/apartment/') && !href.includes('/pad/')) return;
                        let card = a.closest('li, div[class], article') || a.parentElement;
                        if (!card) return;
                        if (!card) return;
                        const text = card.innerText || '';
                        if (text.length < 30 || text.length > 2000) return;
                        const pm = text.match(/\\$[\\d,]+/);
                        if (!pm) return;
                        seen.add(href);
                        results.push({url: href, title: a.innerText.trim().substring(0,200), card_text: text.substring(0,500), price: pm[0]});
                    });
                    return results;
                }
                """)
                print(f'    Found {len(items)} listing links')

                for item in items:
                    pn, pp, em = parse_price(item['price'])
                    card = item['card_text']
                    boro = 'Manhattan' if 'Manhattan' in hood else ('Brooklyn' if 'Brooklyn' in hood else '')
                    zm_rows.append({
                        'source': 'Zumper', 'title': item['title'][:200] or f'Zumper {hood}',
                        'price_raw': item['price'],
                        'price_num': pn, 'price_period': pp or 'month', 'est_monthly': em,
                        'neighborhood': hood, 'borough': boro,
                        'bedrooms': detect_beds(card), 'furnished': detect_furnished(card),
                        'listing_type': 'Rental',
                        'poster_type': 'Likely Landlord/Broker',
                        'description': card[:300], 'url': item['url'], 'scraped_at': now_iso(),
                    })
            except Exception as e:
                print(f'    ❌ {e}')
            polite_sleep(3, 6)

    except Exception as e:
        print(f'  ❌ Zumper error: {e}')
    finally:
        safe_close_page(pg_zm)

    ALL_RESULTS.extend(zm_rows)
    print(f'\n✅ Zumper: {len(zm_rows)} listings')




## 🔑 Loftey (authenticated)


In [ ]:
if PREFLIGHT_ENABLED and not site_preflight_ok('Loftey'):
    print("⏭ Loftey skipped due to preflight status: " + site_preflight_status('Loftey'))
    lf_rows = []
else:
    # ═══════════════════════════════════════
    # Cell 7d: Loftey (authenticated, Playwright)
    # ═══════════════════════════════════════
    ensure_playwright_browser()

    print('  Logging into Loftey...')
    lf_rows = []
    try:
        pg_lf = ctx.new_page()
        pg_lf.goto('https://loftey.com/login', timeout=30000, wait_until='domcontentloaded')
        pg_lf.wait_for_timeout(3000)

        email_in = pg_lf.query_selector('input[type="email"], input[name="email"], input[name="username"], input[placeholder*="email" i]')
        pass_in = pg_lf.query_selector('input[type="password"]')
        if not email_in:
            for inp in pg_lf.query_selector_all('input'):
                t = (inp.get_attribute('type') or '').lower()
                n = (inp.get_attribute('name') or '').lower()
                p = (inp.get_attribute('placeholder') or '').lower()
                if t in ('text','email') or 'email' in n or 'email' in p or 'user' in n:
                    email_in = inp; break

        if email_in and pass_in:
            email_in.fill(CREDS['loftey']['email'])
            pg_lf.wait_for_timeout(300)
            pass_in.fill(CREDS['loftey']['password'])
            pg_lf.wait_for_timeout(300)
            submit = pg_lf.query_selector('button[type="submit"], input[type="submit"], button:has-text("Log"), button:has-text("Sign")')
            if submit: submit.click()
            else: pass_in.press('Enter')
            pg_lf.wait_for_timeout(5000)
            print(f'    Logged in. URL: {pg_lf.url[:60]}')
            require_logged_in(pg_lf, 'Loftey', success_selectors=['a[href*="logout"]', 'a:has-text("Logout")', 'a:has-text("Account")'], success_text=['logout', 'account', 'saved'], stage='loftey_login')
        else:
            raise RuntimeError('Loftey login form not found')

        # Loftey NYC rentals
        LF_SEARCHES = [
            ("Hell's Kitchen", 'https://loftey.com/apartments-for-rent/manhattan/hells-kitchen'),
            ('Midtown West', 'https://loftey.com/apartments-for-rent/manhattan/midtown-west'),
            ('Chelsea', 'https://loftey.com/apartments-for-rent/manhattan/chelsea'),
        ]

        # Secondary source: Loftey is keyword-only for short-term/furnished signals.
        for hood, url in LF_SEARCHES:
            print(f'  Loftey {hood}...')
            try:
                pg_lf.goto(url, timeout=20000, wait_until='domcontentloaded')
                pg_lf.wait_for_timeout(4000)

                items = pg_lf.evaluate("""
                () => {
                    const results = [];
                    const seen = new Set();
                    document.querySelectorAll('a[href]').forEach(a => {
                        const href = a.href;
                        if (seen.has(href)) return;
                        if (!href.includes('loftey.com')) return;
                        if (!href.includes('/property/') && !href.includes('/apartment') && !href.includes('/listing') && !href.includes('/rental')) return;
                        if (href.includes('/login') || href.includes('/about') || href.includes('/blog')) return;
                        let card = a.closest('li, div[class], article') || a.parentElement;
                        if (!card) return;
                        const text = card.innerText || '';
                        if (text.length < 30 || text.length > 2000) return;
                        const pm = text.match(/\\$[\\d,]+/);
                        if (!pm) return;
                        seen.add(href);
                        results.push({url: href, title: a.innerText.trim().substring(0,200), card_text: text.substring(0,500), price: pm[0]});
                    });
                    return results;
                }
                """)
                print(f'    Found {len(items)} listing links')

                for item in items:
                    pn, pp, em = parse_price(item['price'])
                    card = item['card_text']
                    boro = 'Manhattan' if hood in ["Hell's Kitchen",'Upper West Side','Midtown West','Chelsea'] else 'Brooklyn'
                    lf_rows.append({
                        'source': 'Loftey', 'title': item['title'][:200] or f'Loftey {hood}',
                        'price_raw': item['price'],
                        'price_num': pn, 'price_period': pp or 'month', 'est_monthly': em,
                        'neighborhood': hood, 'borough': boro,
                        'bedrooms': detect_beds(card), 'furnished': detect_furnished(card),
                        'listing_type': 'Rental',
                        'poster_type': 'Likely Landlord/Broker',
                        'description': card[:300], 'url': item['url'], 'scraped_at': now_iso(),
                    })
            except Exception as e:
                print(f'    ❌ {e}')
            polite_sleep(3, 6)

    except Exception as e:
        print(f'  ❌ Loftey error: {e}')
    finally:
        safe_close_page(pg_lf)

    ALL_RESULTS.extend(lf_rows)
    print(f'\n✅ Loftey: {len(lf_rows)} listings')




## 🌺 Ohana (authenticated)


In [ ]:
if PREFLIGHT_ENABLED and not site_preflight_ok('Ohana'):
    print("⏭ Ohana skipped due to preflight status: " + site_preflight_status('Ohana'))
    oh_rows = []
else:
    # ═══════════════════════════════════════
    # Cell 10: Ohana — Verified tenant sublets (authenticated)
    # ═══════════════════════════════════════
    # Bubble.io SPA — text-node extraction approach
    # proven working from webarchive analysis.

    ensure_playwright_browser()

    OHANA_URL = 'https://liveohana.ai/sublet/new-york-city'
    oh_rows = []
    pg_oh = ctx.new_page()

    try:
        # Login to Ohana first
        print('  Logging into Ohana...')
        try:
            pg_oh.goto('https://liveohana.ai', timeout=30000, wait_until='domcontentloaded')
            pg_oh.wait_for_timeout(3000)
            login_link = pg_oh.query_selector('a:has-text("Log in"), a:has-text("Sign in"), a:has-text("Login"), button:has-text("Log in")')
            if login_link and login_link.is_visible():
                login_link.click()
                pg_oh.wait_for_timeout(3000)
            else:
                pg_oh.goto('https://liveohana.ai/login', timeout=20000, wait_until='domcontentloaded')
                pg_oh.wait_for_timeout(3000)

            email_in = pg_oh.query_selector('input[type="email"], input[name="email"], input[placeholder*="email" i]')
            pass_in = pg_oh.query_selector('input[type="password"]')
            if not email_in:
                for inp in pg_oh.query_selector_all('input'):
                    t = (inp.get_attribute('type') or '').lower()
                    n = (inp.get_attribute('name') or '').lower()
                    p = (inp.get_attribute('placeholder') or '').lower()
                    if t in ('text','email') or 'email' in n or 'email' in p:
                        email_in = inp; break

            if email_in and pass_in:
                email_in.fill(CREDS['ohana']['email'])
                pg_oh.wait_for_timeout(300)
                pass_in.fill(CREDS['ohana']['password'])
                pg_oh.wait_for_timeout(300)
                submit = pg_oh.query_selector('button[type="submit"], input[type="submit"], button:has-text("Log"), button:has-text("Sign")')
                if submit: submit.click()
                else: pass_in.press('Enter')
                pg_oh.wait_for_timeout(5000)
                print(f'    Logged in. URL: {pg_oh.url[:60]}')
                require_logged_in(pg_oh, 'Ohana', success_selectors=['a[href*="logout"]', 'button:has-text("Log out")', 'a:has-text("Saved")'], success_text=['log out', 'saved', 'messages'], stage='ohana_login')
            else:
                raise RuntimeError('Ohana login form not found')
        except Exception as e:
            raise RuntimeError(f'Ohana login failed: {e}')

        for pg_num in range(3):  # up to 3 pages
            print(f'  Ohana page {pg_num+1}...')
            try:
                if pg_num == 0:
                    pg_oh.goto(OHANA_URL, timeout=45000, wait_until='networkidle')
                    pg_oh.wait_for_timeout(6000)

                items = pg_oh.evaluate("""
                () => {
                    const results = [];
                    const priceRe = /^\$[\d,]+\/mo$/;
                    const dateRe = /(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d|^Now\s*-/;
                    const titleRe = /'s (?:apartment|room) in/;

                    const allEls = [...document.querySelectorAll('*')];
                    const priceEls = allEls.filter(el =>
                        el.children.length === 0 && priceRe.test(el.textContent.trim())
                    );

                    for (const priceEl of priceEls) {
                        let card = priceEl;
                        for (let i = 0; i < 15; i++) {
                            if (!card.parentElement) break;
                            card = card.parentElement;
                            if (card.offsetHeight > 150 && card.offsetWidth > 200) break;
                        }

                        const texts = [...card.querySelectorAll('*')]
                            .filter(el => el.children.length === 0 && el.textContent.trim().length > 1)
                            .map(el => el.textContent.trim());

                        let price = '', dates = '', title = '', beds = '', posted = '';
                        for (const t of texts) {
                            if (priceRe.test(t) && !price) price = t;
                            else if (dateRe.test(t) && !dates) dates = t;
                            else if (titleRe.test(t) && !title) title = t;
                            else if (/^(?:Studio|\d+ Bed)$/i.test(t) && !beds) beds = t;
                            else if (/^Posted/.test(t) && !posted) posted = t;
                        }
                        if (price) results.push({price, dates, title, beds, posted});
                    }

                    const seen = new Set();
                    return results.filter(r => {
                        const key = r.price + '|' + r.title + '|' + r.dates;
                        if (seen.has(key)) return false;
                        seen.add(key); return true;
                    });
                }
                """)
                print(f'    Found {len(items)} listings')

                for item in items:
                    pn, pp, em = parse_price(item['price'])
                    host, location = '', 'New York'
                    m = re.match(r"(.+?)'s\s+(?:apartment|room)\s+in\s+(.+)", item.get('title',''))
                    if m:
                        host = m.group(1).strip()
                        location = m.group(2).strip()
                    borough = 'Brooklyn' if 'Brooklyn' in location else ''
                    is_open = item.get('dates','').endswith('flex')

                    oh_rows.append({
                        'source': 'Ohana',
                        'title': item.get('title') or f'Sublet in {location}',
                        'price_raw': item['price'],
                        'price_num': pn, 'price_period': 'month', 'est_monthly': em,
                        'neighborhood': location, 'borough': borough,
                        'bedrooms': item.get('beds',''),
                        'furnished': 'Yes', 'listing_type': 'Sublet',
                        'poster_type': 'Tenant (sublet)',
                        'dates': item.get('dates',''),
                        'description': f"Ohana sublet from {host}. {item.get('posted','')}. {'Open-ended.' if is_open else ''}",
                        'url': OHANA_URL, 'scraped_at': now_iso(),
                    })

                # Next page
                try:
                    clicked = False
                    for btn in pg_oh.query_selector_all('i, span, button'):
                        txt = btn.inner_text() or ''
                        if 'keyboard_arrow_right' in txt or 'arrow_right' in txt:
                            if btn.is_visible():
                                btn.click()
                                pg_oh.wait_for_timeout(4000)
                                clicked = True
                                break
                    if not clicked: break
                except:
                    break

            except Exception as e:
                print(f'    Error: {e}')
                break
            polite_sleep(3, 5)
    except Exception as e:
        print(f'  ❌ Ohana error: {e}')
    finally:
        safe_close_page(pg_oh)

    ALL_RESULTS.extend(oh_rows)
    print(f'\n✅ Ohana: {len(oh_rows)} listings')


## 🏢 JuneHomes (authenticated)


In [ ]:
if PREFLIGHT_ENABLED and not site_preflight_ok('June Homes'):
    print("⏭ June Homes skipped due to preflight status: " + site_preflight_status('June Homes'))
    jh_rows = []
else:
    # ═══════════════════════════════════════
    # Cell 11: JuneHomes — Furnished flex-lease
    # ═══════════════════════════════════════
    ensure_playwright_browser()

    JH_HOODS = [
        ('NYC Residences', 'https://junehomes.com/residences/new-york-city-ny'),
    ]

    jh_rows = []
    pg_jh = ctx.new_page()

    try:
        # Login to JuneHomes first
        print('  Logging into JuneHomes...')
        try:
            pg_jh.goto('https://junehomes.com/login', timeout=30000, wait_until='domcontentloaded')
            pg_jh.wait_for_timeout(3000)
            email_in = pg_jh.query_selector('input[type="email"], input[name="email"], input[placeholder*="email" i]')
            pass_in = pg_jh.query_selector('input[type="password"]')
            if not email_in:
                for inp in pg_jh.query_selector_all('input'):
                    t = (inp.get_attribute('type') or '').lower()
                    n = (inp.get_attribute('name') or '').lower()
                    p = (inp.get_attribute('placeholder') or '').lower()
                    if t in ('text','email') or 'email' in n or 'email' in p:
                        email_in = inp; break
            if email_in and pass_in:
                email_in.fill(CREDS['jh']['email'])
                pg_jh.wait_for_timeout(300)
                pass_in.fill(CREDS['jh']['password'])
                pg_jh.wait_for_timeout(300)
                submit = pg_jh.query_selector('button[type="submit"], input[type="submit"], button:has-text("Log"), button:has-text("Sign")')
                if submit: submit.click()
                else: pass_in.press('Enter')
                pg_jh.wait_for_timeout(5000)
                print(f'    Logged in. URL: {pg_jh.url[:60]}')
                require_logged_in(pg_jh, 'June Homes', success_selectors=['a[href*="logout"]', 'a:has-text("Logout")', 'a:has-text("Account")'], success_text=['logout', 'account', 'saved'], stage='junehomes_login')
            else:
                raise RuntimeError('June Homes login form not found')
        except Exception as e:
            raise RuntimeError(f'June Homes login failed: {e}')

        for hood, url in JH_HOODS:
            print(f'  JuneHomes {hood}...')
            try:
                pg_jh.goto(url, timeout=30000, wait_until='networkidle')
                pg_jh.wait_for_timeout(4000)
                items = pg_jh.evaluate("""
                () => {
                    const results = [];
                    const seen = new Set();
                    document.querySelectorAll('a[href*="/residences/"]').forEach(a => {
                        const href = a.href;
                        if (seen.has(href)) return;
                        seen.add(href);
                        let card = a.closest('div[class]') || a.parentElement;
                        const text = (card || a).innerText || '';
                        const pm = text.match(/\$[\d,]+/);
                        results.push({url: href, text: text.substring(0, 500), price: pm ? pm[0] : '', title: a.innerText.trim().substring(0, 200)});
                    });
                    return results;
                }
                """)
                print(f'    Found {len(items)} residence links')
                for item in items:
                    if not item.get('price'): continue
                    pn, pp, em = parse_price(item['price'])
                    jh_rows.append({
                        'source': 'June Homes', 'title': item['title'][:200] or f'JuneHomes {hood}',
                        'price_raw': item['price'],
                        'price_num': pn, 'price_period': 'month', 'est_monthly': em,
                        'neighborhood': hood, 'furnished': 'Yes',
                        'listing_type': 'Flex-lease rental', 'poster_type': 'Landlord (June Homes)',
                        'bedrooms': detect_beds(item['text']),
                        'description': item['text'][:300], 'url': item['url'], 'scraped_at': now_iso(),
                    })
            except Exception as e:
                print(f'    Error: {e}')
            polite_sleep(3, 6)
    finally:
        safe_close_page(pg_jh)

    ALL_RESULTS.extend(jh_rows)
    print(f'\n✅ JuneHomes: {len(jh_rows)} listings')



## 🟠 RentHop (no auth)


In [ ]:
if PREFLIGHT_ENABLED and not site_preflight_ok('RentHop'):
    print("⏭ RentHop skipped due to preflight status: " + site_preflight_status('RentHop'))
    rh_rows = []
else:
    # ═══════════════════════════════════════
    # Cell 7: RentHop
    # ═══════════════════════════════════════
    RH_URLS = [
        ('NYC Sublet', 'https://www.renthop.com/apartments-for-rent/new-york-ny/sublet'),
    ]
    rh_rows = []
    for boro, url in RH_URLS:
        print(f'  RentHop {boro}...')
        try:
            r = requests.get(url, headers=HEADERS, timeout=15)
            print(f'    Status {r.status_code}, {len(r.text)} chars')
            if r.status_code != 200: continue
            raw = universal_extract(r.text, 'https://www.renthop.com')
            raw = [x for x in raw if '/listings/' in x['url'] and len(x['card_text']) > 50]
            print(f'    Listing candidates: {len(raw)}')
            for item in raw:
                pn, pp, em = parse_price(item['price_found'])
                card = item['card_text']
                rh_rows.append({
                    'source': 'RentHop', 'title': item['title'][:200],
                    'price_raw': item['price_found'],
                    'price_num': pn, 'price_period': pp or 'month', 'est_monthly': em,
                    'borough': boro, 'bedrooms': detect_beds(card),
                    'listing_type': 'Sublet/Rental', 'poster_type': 'Likely Landlord/Broker',
                    'description': card[:300], 'url': item['url'], 'scraped_at': now_iso(),
                })
        except Exception as e:
            print(f'    Error: {e}')
        polite_sleep(3, 5)

    ALL_RESULTS.extend(rh_rows)
    print(f'\n✅ RentHop: {len(rh_rows)} listings')


## Listings Project (discovery only)

In [ ]:

if PREFLIGHT_ENABLED and not site_preflight_ok('Listings Project'):
    print("⏭ Listings Project skipped due to preflight status: " + site_preflight_status('Listings Project'))
    lp_rows = []
else:
    ensure_playwright_browser()

    def _lp_first_text(page_obj, selectors):
        for sel in selectors:
            try:
                el = page_obj.query_selector(sel)
                if el and el.is_visible():
                    val = el.inner_text().strip()
                    if val:
                        return val
            except Exception:
                continue
        return ""

    lp_pg = ctx.new_page()
    lp_rows = []

    try:
        print('  Logging into Listings Project...')
        lp_pg.goto('https://www.listingsproject.com/login', timeout=30000, wait_until='domcontentloaded')
        lp_pg.wait_for_timeout(3000)

        email_in = lp_pg.query_selector('input[type="email"], input[name="email"]')
        pass_in = lp_pg.query_selector('input[type="password"]')
        if not email_in:
            for inp in lp_pg.query_selector_all('input'):
                t = (inp.get_attribute('type') or '').lower()
                n = (inp.get_attribute('name') or '').lower()
                p = (inp.get_attribute('placeholder') or '').lower()
                if t in ('text', 'email') or 'email' in n or 'email' in p:
                    email_in = inp
                    break

        if not (email_in and pass_in):
            raise RuntimeError("Could not find LP login form")
        email_in.fill(CREDS['lp']['email'])
        lp_pg.wait_for_timeout(300)
        pass_in.fill(CREDS['lp']['password'])
        lp_pg.wait_for_timeout(300)
        submit = lp_pg.query_selector('button[type="submit"], input[type="submit"], button:has-text("Log in"), button:has-text("Sign in")')
        if submit:
            submit.click()
        else:
            pass_in.press('Enter')
        lp_pg.wait_for_timeout(5000)

        if 'login' in lp_pg.url.lower():
            raise RuntimeError('Listings Project login appears to have failed')
        require_logged_in(lp_pg, 'Listings Project', success_selectors=['a[href*="logout"]', 'button:has-text("Log out")', 'a:has-text("Saved")'], success_text=['log out', 'saved listings', 'account'], stage='listings_project_login')

        print('  Collecting Listings Project sublet URLs...')
        lp_pg.goto('https://www.listingsproject.com/real-estate/new-york-city/sublets', timeout=30000, wait_until='domcontentloaded')
        lp_pg.wait_for_timeout(4000)

        all_listing_urls = set()
        for scroll_round in range(20):
            links = lp_pg.evaluate("""
            () => [...new Set(
                [...document.querySelectorAll('a[href*="/listings/"]')]
                .map(a => a.href.split('#')[0])
                .filter(h => !h.includes('/resources/') && !h.includes('/login'))
            )]
            """)
            before = len(all_listing_urls)
            all_listing_urls.update(links)
            new_count = len(all_listing_urls) - before
            print(f'    Scroll {scroll_round+1}: {len(links)} links ({new_count} new) — total: {len(all_listing_urls)}')
            if new_count == 0 and scroll_round > 2:
                break
            next_btn = lp_pg.query_selector('a:has-text("Next"), button:has-text("Next"), button:has-text("Load more"), [aria-label="Next page"]')
            if next_btn and next_btn.is_visible():
                next_btn.click()
                lp_pg.wait_for_timeout(3000)
            else:
                lp_pg.evaluate('window.scrollTo(0, document.body.scrollHeight)')
                lp_pg.wait_for_timeout(2000)
                after = lp_pg.evaluate("() => document.querySelectorAll('a[href*="/listings/"]').length")
                if after <= len(links):
                    break

        listing_urls = sorted(all_listing_urls)
        print(f'\n✅ {len(listing_urls)} listing URLs collected\n')

        for i, url in enumerate(listing_urls):
            print(f'  [{i+1}/{len(listing_urls)}] {url.split("/listings/")[-1][:55]}...')
            try:
                lp_pg.goto(url, timeout=20000, wait_until='domcontentloaded')
                lp_pg.wait_for_timeout(2000)

                title = _lp_first_text(lp_pg, ['h1', 'h2', '[class*="title"]'])
                price = _lp_first_text(lp_pg, ['[class*="price"]', '[data-testid*="price"]'])
                page_text = normalize_space(lp_pg.locator('body').inner_text(timeout=5000))[:5000]
                address = extract_exact_address(page_text)

                rec = {
                    'source': 'Listings Project',
                    'title': title,
                    'price_raw': price,
                    'price_num': parse_price(price)[0],
                    'price_period': parse_price(price)[1] or 'month',
                    'est_monthly': parse_price(price)[2],
                    'listing_type': 'Sublet',
                    'description': page_text[:1000],
                    'address': address,
                    'url': url,
                    'scraped_at': now_iso(),
                }
                lp_rows.append(rec)
                polite_sleep(1, 3)
            except Exception as e:
                print(f'    ⚠ error: {e}')

        print(f'\n✅ Listings Project discovery complete: {len(lp_rows)} rows')
        ALL_RESULTS.extend(lp_rows)
    except Exception as e:
        print(f'❌ Listings Project discovery failed: {e}')
    finally:
        try:
            lp_pg.close()
        except Exception:
            pass



## Combine + Export

In [ ]:
# ═══════════════════════════════════════
# Combine, Deduplicate, Export
# ═══════════════════════════════════════
before = len(ALL_RESULTS)
if PREFLIGHT_ENABLED:
    print('\nPreflight status summary:')
    for _site in sorted(SOURCE_HEALTH):
        _info = SOURCE_HEALTH.get(_site, {})
        print(f"  {_site:>18}: {_info.get('status','UNTESTED')} ({_info.get('stage','')})")

results = deduplicate([enrich_listing(r) for r in ALL_RESULTS])
dupes = before - len(results)

print(f'Total scraped: {len(results)} unique listings ({dupes} duplicates removed)')

# Alert on sources where preflight PASSed but scraper got zero results
if PREFLIGHT_ENABLED:
    _src_counts = {}
    for _r in results:
        _s = _r.get('source', '?')
        _src_counts[_s] = _src_counts.get(_s, 0) + 1
    for _site in sorted(SOURCE_POLICIES):
        _info = SOURCE_HEALTH.get(_site, {})
        if _info.get('status') == 'PASS' and _src_counts.get(_site, 0) == 0 and SOURCE_POLICIES[_site].get('active'):
            print(f'  \u26a0\ufe0f {_site}: preflight PASS but 0 listings — check scraper selectors')
print(f'Active constraints: <= ${SEARCH_MAX_MONTHLY:,} | {TARGET_AREA_LABEL} | {SEARCH_MIN_MONTHS}-{SEARCH_MAX_MONTHS} months')

goal_results = [r for r in results if r.get('goal_match') == 'Yes']
exact_address_hits = [r for r in goal_results if r.get('action_bucket') == 'exact_address_hit']
contact_first_queue = [r for r in goal_results if r.get('action_bucket') == 'contact_queue']
mixed_visibility_queue = [r for r in goal_results if r.get('action_bucket') == 'manual_review_or_contact']
parser_review_queue = [r for r in goal_results if r.get('action_bucket') == 'parser_review']
constraint_miss_queue = [r for r in results if r.get('goal_match') != 'Yes']

print(f'Discovery candidates: {len(goal_results)}')
print(f'Exact-address hits: {len(exact_address_hits)}')
print(f'Contact-first queue: {len(contact_first_queue)}')
print(f'Mixed-visibility queue: {len(mixed_visibility_queue)}')
print(f'Parser-review queue: {len(parser_review_queue)}')
print(f'Filtered out by constraints: {len(constraint_miss_queue)}')

print('\nCandidate source breakdown:')
src_counts = {}
for r in goal_results:
    s = r.get('source', '?')
    src_counts[s] = src_counts.get(s, 0) + 1
for s, c in sorted(src_counts.items(), key=lambda x: (-x[1], x[0])):
    print(f'  {s:>20}: {c}')

print('\nTop filter-out reasons:')
fail_counts = {}
for r in constraint_miss_queue:
    for reason in [x.strip() for x in (r.get('search_fail_reasons') or '').split(';') if x.strip()]:
        fail_counts[reason] = fail_counts.get(reason, 0) + 1
for reason, c in sorted(fail_counts.items(), key=lambda x: (-x[1], x[0]))[:12]:
    print(f'  {reason:>24}: {c}')

prices = [r['est_monthly'] for r in goal_results if r.get('est_monthly')]
if prices:
    prices_sorted = sorted(prices)
    print(f'\nCandidate price range: ${min(prices):,} - ${max(prices):,}')
    print(f'Candidate median: ${prices_sorted[len(prices_sorted)//2]:,}')

csv_path_all = write_results(results, '01_all_raw_listings.csv')
csv_path_goal = write_results(goal_results, '01_candidates_filtered.csv')
csv_path_exact = write_results(exact_address_hits, '01_exact_address_hits.csv')
csv_path_contact = write_results(contact_first_queue, '01_contact_queue.csv')
csv_path_mixed = write_results(mixed_visibility_queue, '01_mixed_visibility_queue.csv')
csv_path_parser = write_results(parser_review_queue, '01_parser_review.csv')
csv_path_misses = write_results(constraint_miss_queue, '01_rejected_with_reasons.csv')

json_path = OUTPUT_DIR / '01_candidates_filtered.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(goal_results, f, indent=2, default=str)
print(f'  -> JSON: {json_path}')

goal_df = pd.DataFrame(goal_results)
preview_cols = [
    'source', 'title', 'price_raw', 'est_monthly', 'neighborhood', 'address',
    'duration_months_min', 'duration_months_max', 'action_bucket', 'url'
]
preview_cols = [c for c in preview_cols if c in goal_df.columns]
if len(goal_df):
    display(goal_df[preview_cols].head(25))
else:
    print('No listings matched all active constraints.')


run_log = {
    'scraped_total_unique': len(results),
    'candidates_filtered': len(goal_results),
    'exact_address_hits': len(exact_address_hits),
    'contact_queue': len(contact_first_queue),
    'mixed_visibility_queue': len(mixed_visibility_queue),
    'parser_review_queue': len(parser_review_queue),
    'rejected': len(constraint_miss_queue),
    'generated_at': now_iso(),
    'constraints': {
        'max_monthly': SEARCH_MAX_MONTHLY,
        'min_months': SEARCH_MIN_MONTHS,
        'max_months': SEARCH_MAX_MONTHS,
        'target_area': TARGET_AREA_LABEL,
    },
    'source_health': SOURCE_HEALTH,
}
run_log_path = OUTPUT_DIR / '01_run_log.json'
with open(run_log_path, 'w', encoding='utf-8') as f:
    json.dump(run_log, f, indent=2, default=str)
print(f'  -> JSON: {run_log_path}')


In [ ]:
# ═══════════════════════════════════════
# Save outputs to Google Drive
# ═══════════════════════════════════════
import shutil

if SAVE_TO_DRIVE:
    ts = datetime.now().strftime('%Y%m%d_%H%M')
    copied = 0
    for src in sorted(OUTPUT_DIR.glob('01_*.*')):
        if src.is_file():
            shutil.copy2(src, DRIVE_DIR / src.name)
            shutil.copy2(src, DRIVE_DIR / f'{src.stem}_{ts}{src.suffix}')
            copied += 1
    print(f'✅ Saved {copied} output files to {DRIVE_DIR}')
else:
    print(f'Drive save skipped. Outputs remain at: {OUTPUT_DIR}')
    if running_in_colab():
        latest_csv = OUTPUT_DIR / '01_candidates_filtered.csv'
        if latest_csv.exists():
            from google.colab import files
            files.download(str(latest_csv))


In [ ]:
# ═══════════════════════════════════════
# Cell 12: Optional close Playwright (run LAST)
# ═══════════════════════════════════════
AUTO_CLOSE_PLAYWRIGHT_NOW = False  # Set True only when all browser-based cells are finished.

if AUTO_CLOSE_PLAYWRIGHT_NOW:
    close_playwright_browser()
    print('✅ Playwright closed')
else:
    print('ℹ️ Skipped browser shutdown so later browser cells can still run. Set AUTO_CLOSE_PLAYWRIGHT_NOW = True when you are completely done.')

## 📱 Download Results + Diagnostics

In [ ]:
# ═══════════════════════════════════════
# Download Results + Diagnostics to Phone
# Run this as the LAST cell after Cell 33
# ═══════════════════════════════════════
import zipfile, json, os
from pathlib import Path
from datetime import datetime

OUTPUT_DIR = Path('output')
ZIP_NAME = f'nyc_listings_{datetime.now().strftime("%Y%m%d_%H%M")}.zip'

# --- Build diagnostics summary ---
diag = []
diag.append("=== RUN DIAGNOSTICS ===\n")

# Source health
if os.path.exists(OUTPUT_DIR / '01_run_log.json'):
    with open(OUTPUT_DIR / '01_run_log.json') as f:
        log = json.load(f)
    diag.append(f"Total scraped (unique): {log.get('scraped_total_unique', '?')}")
    diag.append(f"Candidates filtered:   {log.get('candidates_filtered', '?')}")
    diag.append(f"Exact address hits:    {log.get('exact_address_hits', '?')}")
    diag.append(f"Contact queue:         {log.get('contact_queue', '?')}")
    diag.append(f"Mixed visibility:      {log.get('mixed_visibility_queue', '?')}")
    diag.append(f"Parser review:         {log.get('parser_review_queue', '?')}")
    diag.append(f"Rejected:              {log.get('rejected', '?')}")
    diag.append(f"Generated at:          {log.get('generated_at', '?')}")
    diag.append(f"\nConstraints: {json.dumps(log.get('constraints', {}), indent=2)}")

    diag.append("\n\n=== SOURCE HEALTH ===\n")
    health = log.get('source_health', {})
    for src in sorted(health):
        info = health[src]
        status = info.get('status', '?')
        stage = info.get('stage', '')
        details = info.get('details', '')[:80]
        diag.append(f"  {src:>18}: {status:<10} {stage:<18} {details}")

# Auth session results
try:
    if AUTH_SESSION_LOG:
        diag.append("\n\n=== AUTH SESSION LOG ===\n")
        for src, info in sorted(AUTH_SESSION_LOG.items()):
            ok = '✅' if info.get('ok') else '❌'
            diag.append(f"  {src:>18}: {ok}  url={info.get('url', '?')[:50]}")
            diag.append(f"                     selector_hit={info.get('selector_hit')}  text_hit={info.get('text_hit')}  has_password={info.get('has_password')}")
except NameError:
    diag.append("\n(AUTH_SESSION_LOG not available)")

# Per-source row counts from results
try:
    if results:
        diag.append("\n\n=== PER-SOURCE ROW COUNTS ===\n")
        from collections import Counter
        src_counts = Counter(r.get('source', '?') for r in results)
        goal_counts = Counter(r.get('source', '?') for r in results if r.get('goal_match') == 'Yes')
        for src in sorted(src_counts):
            diag.append(f"  {src:>18}: {src_counts[src]:3d} total, {goal_counts.get(src, 0):3d} matched")
except NameError:
    pass

# Top rejection reasons
try:
    if results:
        diag.append("\n\n=== TOP REJECTION REASONS ===\n")
        reason_counts = Counter()
        for r in results:
            if r.get('goal_match') != 'Yes':
                for reason in (r.get('search_fail_reasons') or '').split(';'):
                    reason = reason.strip()
                    if reason:
                        reason_counts[reason] += 1
        for reason, count in reason_counts.most_common(15):
            diag.append(f"  {count:4d}  {reason}")
except NameError:
    pass

# Preflight details
if os.path.exists(OUTPUT_DIR / 'nyc_preflight_summary.csv'):
    diag.append("\n\n=== PREFLIGHT SUMMARY ===\n")
    import csv
    with open(OUTPUT_DIR / 'nyc_preflight_summary.csv') as f:
        for row in csv.DictReader(f):
            diag.append(f"  {row.get('source', '?'):>18}: {row.get('status', '?'):<10} {row.get('stage', ''):<18} {(row.get('details', '') or '')[:60]}")

diag_text = '\n'.join(diag)
diag_path = OUTPUT_DIR / 'diagnostics.txt'
with open(diag_path, 'w') as f:
    f.write(diag_text)
print(diag_text)

# --- Zip everything ---
with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUTPUT_DIR.glob('01_*.*')):
        zf.write(f, f.name)
    zf.write(diag_path, 'diagnostics.txt')
    # Include preflight summary if exists
    for f in OUTPUT_DIR.glob('nyc_preflight_*.*'):
        zf.write(f, f'preflight/{f.name}')

zip_size = os.path.getsize(ZIP_NAME)
print(f"\n{'='*50}")
print(f"📦 {ZIP_NAME} ({zip_size:,} bytes)")
print(f"{'='*50}")

# List contents
with zipfile.ZipFile(ZIP_NAME) as zf:
    for info in zf.infolist():
        print(f"  {info.compress_size:>8,} bytes  {info.filename}")

# Download
try:
    from google.colab import files
    files.download(ZIP_NAME)
    print(f"\n📱 Download started — check your browser/phone downloads")
except ImportError:
    print(f"\n(Not in Colab — zip is at: {os.path.abspath(ZIP_NAME)})")
